# Multi-Modal Architecture Comparison for Daily Commodity Direction Prediction with News and Price Signals

### A Component-Level Study of Graph, Geometry, and Attention Design Choices

---

## Abstract

We study the daily-frequency direction-prediction problem on commodity futures using two information sources: technical indicators derived from price/volume and news-sentiment embeddings. We compare nine architectures spanning three families: classical machine learning baselines (logistic regression, gradient-boosted trees, ARIMA-X), deep sequence models (LSTM, LSTM with causal self-attention, cross-attention, dual-stream attention), and a graph-spectral method that we adapt from the directed-graph signal-processing literature (Defferrard et al., 2016; Tong et al., 2020; Rey et al., 2025). The graph-spectral architecture, which we call **CD-GSHA** (Causal Directional Graph-Spectral Hyperbolic Attention), constructs a strictly lower-triangular news graph, applies Chebyshev convolution under random-walk normalization, and combines Euclidean and hyperbolic similarity in a learnable hybrid attention.

We apply identical training, evaluation, and backtesting protocols across all architectures on three commodities (wheat, corn, crude oil) over approximately 17 years of daily data. We report standard classification metrics, but also a full long-short backtest with confidence-weighted position sizing, transaction-cost sensitivity, Sharpe and Sortino ratios, drawdown, turnover, and statistical significance tests (Diebold-Mariano on forecast errors, Pesaran-Timmermann on directional accuracy).

This notebook implements the wheat experiment end to end. The corn and crude-oil experiments use identical code with different data files.

The paper's contributions are: (1) a rigorous, component-level comparison of multi-modal architectures on weak-signal commodity prediction, (2) ablations decomposing the contribution of graph structure, geometry, and attention design, and (3) trading-relevant evaluation that distinguishes statistical accuracy from economic value.

---


## 1. Problem Setup

### 1.1 Task

At each trading day $t$ we predict whether the next-day settlement price will be higher or lower than today's:

$$y_{t+1} = \mathbf{1}[P_{t+1} > P_t] \in \{0, 1\}.$$

This is a binary classification task. The model never sees tomorrow's price. The signal is weak: for daily commodity direction, even strong models rarely exceed F1 of 0.55–0.60.

### 1.2 Inputs

For each example, we use a 30-day lookback window ending on day $t$:

- **Price features** $\mathbf{X} \in \mathbb{R}^{30 \times 19}$ — 19 daily technical and calendar features (defined in §3.2)
- **News embeddings** $\mathbf{E} \in \mathbb{R}^{30 \times 16}$ — 16-D FinBERT embedding of each day's news, lagged by one calendar day; rows are zero on news-absent days
- **News mask** $\mathbf{m} \in \{0,1\}^{30}$ — indicator of news availability per day

Approximately half of trading days carry no news, so $\mathbf{m}$ is a first-class input rather than an afterthought.

### 1.3 Why this setting is hard

Three structural difficulties:

1. **Weak signal.** Daily direction sits near 50% chance. Most days carry no exploitable directional information.
2. **Sparse, irregularly-arriving news.** Different days have very different amounts of news context.
3. **Cross-modal asymmetry.** Price is sequential and dense; news is structured and sparse. The standard transformer/LSTM approach treats news as just more vector input, which discards the relational structure between events.

### 1.4 What we compare

Three architecture families, nine total models, plus seven ablation variants of CD-GSHA:

**Classical baselines (3):** Logistic regression on engineered features, XGBoost, ARIMA-X with sentiment as exogenous variable.

**Deep baselines (4):** LSTM, LSTM + causal self-attention, cross-attention (price queries news), dual-stream attention (parallel self-attention on each modality).

**Graph-spectral methods (2):** A symmetric-graph variant (re-implementing the standard Chebyshev-on-symmetric-graph approach from Defferrard et al. 2016, applied to a learned news graph), and CD-GSHA (the directional variant described in §6).

**Ablations of CD-GSHA (7):** Removing temporal priors, the graph itself, hyperbolic component, Euclidean component, price-side causal attention, swapping directional graph for symmetric graph, and a multi-head variant.

All models train on identical features, splits, loss, optimizer, and ensemble protocol. The only difference between any two models is the architecture.


---

## 2. Mathematical Preliminaries

### 2.1 Causal directional graph

We construct a graph $G_t = (V_t, A_t)$ over the 30 days in the lookback window. Edges flow strictly from past to present. For nodes $i, j$ corresponding to days in the window:

**Causal semantic edges** (between news-present days):
$$
A^{\text{sem}}_{ij} =
\begin{cases}
m_i\,m_j\,\sigma\!\big(\kappa\,(s_{ij} - \tau)\big) & \text{if } j \le i \\
0 & \text{if } j > i,
\end{cases}
\qquad s_{ij} = \frac{\mathbf{e}_i^\top \mathbf{e}_j}{\|\mathbf{e}_i\|\,\|\mathbf{e}_j\|}.
$$
where $\kappa = 10$ is a fixed sharpness and $\tau$ is a learnable similarity threshold.

**Causal temporal-prior edges** (always on, decaying with separation):
$$
A^{\text{tmp}}_{ij} =
\begin{cases}
\exp(-\gamma\,(i - j)) & \text{if } j \le i \\
0 & \text{if } j > i.
\end{cases}
$$

**Self-loops** (learnable weight):
$$
A^{\text{self}}_{ii} = \delta = \mathrm{softplus}(\theta_\delta) \ge 0, \qquad A^{\text{self}}_{ij} = 0 \text{ for } i \neq j.
$$

**Combined adjacency:**
$$
A_{ij} = \begin{cases}
\delta & j = i \\
\alpha_g A^{\text{sem}}_{ij} + \beta_g A^{\text{tmp}}_{ij} & j < i \\
0 & j > i,
\end{cases}
$$
with all of $\{\tau, \gamma, \alpha_g, \beta_g, \delta\}$ learnable. By construction $A$ is lower-triangular (including the diagonal).

### 2.2 Random-walk normalization for directed graphs

The symmetric normalization $\mathbf{D}^{-1/2}\mathbf{A}\mathbf{D}^{-1/2}$ requires $\mathbf{A} = \mathbf{A}^\top$ and is undefined for our directed $A$. We use the random-walk normalization, which is the canonical choice for directed graphs (Chung 2005):
$$
\hat{\mathbf{L}}^{\text{dir}} = \mathbf{I} - \mathbf{D}_{\text{out}}^{-1}\mathbf{A}, \qquad
(D_{\text{out}})_{ii} = \max\!\Big(\sum_j A_{ij},\, \varepsilon\Big).
$$

Two properties matter for what follows:
1. **Lower-triangularity is preserved.** $\mathbf{D}_{\text{out}}^{-1}\mathbf{A}$ has the same sparsity pattern as $\mathbf{A}$, so $\hat{\mathbf{L}}^{\text{dir}}$ is lower-triangular.
2. **Spectrum is bounded.** Each row of $\mathbf{D}_{\text{out}}^{-1}\mathbf{A}$ sums to 1 (or 0 for isolated nodes), so $\rho(\mathbf{D}_{\text{out}}^{-1}\mathbf{A}) \le 1$ and the eigenvalues of $\hat{\mathbf{L}}^{\text{dir}}$ lie in $[0, 2]$, the same range required for stable Chebyshev approximation (Defferrard et al. 2016).

### 2.3 Causal Chebyshev spectral convolution

The order-$K$ Chebyshev expansion uses the standard recursion:
$$
T_0(\hat{\mathbf{L}}^{\text{dir}}) = \mathbf{I},\qquad
T_1(\hat{\mathbf{L}}^{\text{dir}}) = \hat{\mathbf{L}}^{\text{dir}},\qquad
T_{k+1} = 2\hat{\mathbf{L}}^{\text{dir}} T_k - T_{k-1}.
$$

Because the product of lower-triangular matrices is lower-triangular, every $T_k(\hat{\mathbf{L}}^{\text{dir}})$ is lower-triangular by induction. Applied to news embeddings $\mathbf{E}$, this gives:
$$
\big[T_k(\hat{\mathbf{L}}^{\text{dir}})\,\mathbf{E}\big]_i = \sum_{j \le i} [T_k]_{ij}\,\mathbf{e}_j.
$$

The output at day $i$ depends only on news at days $j \le i$ — the convolution is causal by graph topology, not by post-hoc masking. The full convolution layer is:
$$
\mathbf{Z} = \mathrm{LN}\!\Big(\mathrm{GELU}\!\Big(\sum_{k=0}^{K-1} T_k(\hat{\mathbf{L}}^{\text{dir}})\mathbf{E}\mathbf{W}_k\Big)\Big) + \eta_r\mathbf{E}\mathbf{W}_r,
$$
with $K = 3$, residual scale $\eta_r = 0.3$.

### 2.4 Poincaré ball and Möbius operations

Hyperbolic geometry is well-suited to embedding hierarchies because volume grows exponentially with radius (Nickel & Kiela 2017). The Poincaré ball of curvature $c > 0$ is
$$
\mathbb{B}_c^n = \{\mathbf{x} \in \mathbb{R}^n : c\|\mathbf{x}\|^2 < 1\}.
$$

**Möbius addition** plays the role of vector addition:
$$
\mathbf{x} \oplus_c \mathbf{y} = \frac{(1 + 2c\langle\mathbf{x},\mathbf{y}\rangle + c\|\mathbf{y}\|^2)\mathbf{x} + (1 - c\|\mathbf{x}\|^2)\mathbf{y}}{1 + 2c\langle\mathbf{x},\mathbf{y}\rangle + c^2\|\mathbf{x}\|^2\|\mathbf{y}\|^2}.
$$

**Exponential map at the origin** lifts a Euclidean tangent vector to the ball:
$$
\exp_\mathbf{0}^c(\mathbf{v}) = \tanh(\sqrt{c}\,\|\mathbf{v}\|)\,\frac{\mathbf{v}}{\sqrt{c}\,\|\mathbf{v}\|}.
$$

**Squared Poincaré distance** (used instead of unsquared distance for numerical stability — avoids $\sqrt{\cdot}$ near zero):
$$
d_c^2(\mathbf{x}, \mathbf{y}) = \frac{4}{c}\,\operatorname{arctanh}^2(\sqrt{c}\,\|{-\mathbf{x}} \oplus_c \mathbf{y}\|).
$$

In implementation, all points are clipped to $\|\mathbf{x}\| \le (1 - 10^{-3})/\sqrt{c}$ to avoid the boundary singularity.

### 2.5 Hybrid hyperbolic-Euclidean attention

Pure hyperbolic attention $\propto \exp(-d_c)$ is fragile near the ball boundary because $\operatorname{arctanh}$ saturates and gradients vanish. We combine Euclidean dot-product with negative squared Poincaré distance, with learnable mixing:
$$
e_i = \underbrace{\frac{\alpha}{\sqrt{d_h}}\,\mathbf{q}'^\top\mathbf{k}_i'}_{\text{Euclidean}} \;-\; \underbrace{\beta\,c\,d_c^2(\tilde{\mathbf{q}}, \tilde{\mathbf{k}}_i)}_{\text{Hyperbolic}}.
$$
$\alpha = \mathrm{softplus}(\theta_\alpha), \beta = \mathrm{softplus}(\theta_\beta) \ge 0$ are mixing coefficients, $c = \mathrm{softplus}(\theta_c) + 0.1$ is the curvature with a floor at 0.1. At initialization $\alpha \approx 0.69$, $\beta \approx 0.10$, $c = 0.5$ — near-pure Euclidean. The hyperbolic correction grows during training only if it lowers training loss.

We use a single attention head ($H = 1$). On weak-signal direction prediction tasks, multi-head splits the representation into low-dimensional subspaces below the capacity needed for a useful attention pattern; this is confirmed empirically in our ablations.

### 2.6 Cross-modal confidence-gated fusion

Combining price context $\mathbf{p}$ and news context $\mathbf{c}$ via a per-dimension gate conditioned on news density $\hat{m} = \frac{1}{T}\sum_i m_i$:
$$
\boldsymbol{\rho} = \sigma\!\big(\mathbf{W}_2\,\mathrm{GELU}(\mathbf{W}_1[\mathbf{p};\mathbf{c};\hat{m}] + \mathbf{b}_1) + \mathbf{b}_2\big), \qquad
\mathbf{f} = \boldsymbol{\rho} \odot \mathbf{c} + (\mathbf{1} - \boldsymbol{\rho}) \odot \mathbf{p}.
$$
The gate is per-dimension (not scalar), letting different feature dimensions independently weight news against price.

### 2.7 Training objective

Class-balanced binary focal loss with label smoothing (Lin et al. 2017):
$$
\mathcal{L}(z, y) = -w_+(y)\,(1 - p_t)^\gamma\,\log p_t,\qquad
p_t = \begin{cases}\sigma(z) & y \ge 0.5 \\ 1-\sigma(z) & y < 0.5\end{cases}
$$
with $\gamma = 2$, $\varepsilon_{\text{smooth}} = 0.05$, and $w_+ = (1-\pi)/\pi$ from the train positive rate $\pi$.

---


## 3. Setup and Data Pipeline

### 3.1 Environment and reproducibility

We seed Python, NumPy, PyTorch (CPU+CUDA), and set CuDNN to deterministic mode. Each training run also calls `set_seed` with its own seed.


In [7]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'torch', 'scikit-learn', 'numpy', 'pandas',
                       'matplotlib', 'seaborn', 'xgboost', 'statsmodels'])

import os, random, copy, time, warnings, math
warnings.filterwarnings('ignore')

import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.swa_utils import AveragedModel
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (f1_score, accuracy_score, recall_score,
                             precision_score, confusion_matrix, roc_auc_score,
                             matthews_corrcoef, balanced_accuracy_score)
import xgboost as xgb
from statsmodels.tsa.arima.model import ARIMA
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device = {DEVICE}')


PyTorch 2.10.0 | device = cpu



[notice] A new release of pip is available: 24.2 -> 26.1
[notice] To update, run: pip install --upgrade pip


### 3.2 Configuration and constants

Single source of truth for paths, lookback length, and embedding dimension. Change `COMMODITY` and the data paths to switch between wheat / corn / oil — everything else is identical across commodities.


In [8]:
COMMODITY = 'wheat'
DATA_DIR  = 'data'

# PRICE_FILE     = f'{DATA_DIR}/{COMMODITY}_prices.csv'
# SENTIMENT_FILE = f'{DATA_DIR}/{COMMODITY}_daily_news_sentiment.csv'
# EMBEDDING_FILE = f'{DATA_DIR}/{COMMODITY}_daily_news_embeddings.pt'

base_path = 'data'
PRICE_FILE = f'{base_path}/wheat_prices.csv'
SENTIMENT_FILE = f'{base_path}/wheat_daily_news_sentiment.csv'
EMBEDDING_FILE = f'{base_path}/wheat_daily_news_embeddings.pt'

LOOKBACK = 30          # days of history per example
EMB_DIM  = 16          # FinBERT embedding dimension

# Numerical stability constants for hyperbolic ops
EPS      = 1e-6
MAX_NORM = 1.0 - 1e-3


### 3.3 Load raw data

News from calendar day $d$ is treated as available from day $d+1$ onwards (one-day lag, no same-day leakage). News in our dataset comes from SerpAPI Google News results, encoded with FinBERT.


In [9]:
# ── Price data ─────────────────────────────────────────────────────────
price_df = pd.read_csv(PRICE_FILE)
price_df['Date'] = pd.to_datetime(price_df['Date'])
price_df = price_df.sort_values('Date').set_index('Date')

# Strip thousands separators; some CSVs save numbers as "1,234.56"
for col in ['Price', 'Open', 'High', 'Low']:
    price_df[col] = price_df[col].replace({',': ''}, regex=True).astype(float)

def _parse_volume(v):
    '''Volume strings come as "1.23M" or "456.78K"; normalise to float.'''
    if pd.isna(v) or str(v).strip() in ('', '-'):
        return np.nan
    v = str(v).strip().replace(',', '')
    if v.endswith('K'): return float(v[:-1]) * 1_000
    if v.endswith('M'): return float(v[:-1]) * 1_000_000
    return float(v)
price_df['Volume'] = price_df['Vol.'].apply(_parse_volume)

# ── News sentiment + embeddings ───────────────────────────────────────
sentiment_df  = pd.read_csv(SENTIMENT_FILE)
sentiment_map = dict(zip(sentiment_df['date'], sentiment_df['sentiment_score']))
news_emb      = torch.load(EMBEDDING_FILE, map_location='cpu', weights_only=False)

print(f'Price rows     : {len(price_df)}')
print(f'Sentiment days : {len(sentiment_map)}')
print(f'Embedding days : {len(news_emb)}')
print(f'Date range     : {price_df.index.min().date()} → {price_df.index.max().date()}')


FileNotFoundError: [Errno 2] No such file or directory: 'data/wheat_daily_news_sentiment.csv'

### 3.4 Feature engineering

Nineteen features per day across price, return, volatility, momentum, range, volume, calendar, and sentiment.

Three design choices worth noting:

1. **Log returns instead of raw prices.** Price levels trend over years and are non-stationary; log returns are approximately stationary and scale-invariant.
2. **Day-of-week as $(\sin, \cos)$.** Using the integer 0–4 makes Friday (4) and Monday (0) appear maximally distant when in trading-week terms they are adjacent. The trigonometric encoding fixes this.
3. **Strict no-look-ahead in the sentiment lookup.** `decayed_sentiment(d)` only returns sentiment from days $\le d - 1$. Same-day sentiment is forbidden.


In [ ]:
# Returns and volatility
price_df['Return']     = np.log(price_df['Price'] / price_df['Price'].shift(1))
price_df['Volatility'] = price_df['Return'].rolling(5).std()

# RSI(14)
delta = price_df['Price'].diff()
gain  = delta.where(delta > 0, 0.0).rolling(14).mean()
loss  = (-delta.where(delta < 0, 0.0)).rolling(14).mean()
price_df['RSI'] = 100 - (100 / (1 + gain / loss.replace(0, np.nan)))

# MACD (12 EMA - 26 EMA)
ema12 = price_df['Price'].ewm(span=12, adjust=False).mean()
ema26 = price_df['Price'].ewm(span=26, adjust=False).mean()
price_df['MACD'] = ema12 - ema26

# Bollinger %B (position within 2σ bands around 20-day SMA)
bb_mid = price_df['Price'].rolling(20).mean()
bb_std = price_df['Price'].rolling(20).std()
price_df['BB_pctB'] = (price_df['Price'] - (bb_mid - 2*bb_std)) / (4*bb_std)

# Volume change %
price_df['Volume']  = price_df['Volume'].ffill().bfill()
price_df['Vol_chg'] = price_df['Volume'].pct_change().replace([np.inf, -np.inf], np.nan).fillna(0.0)

# Multi-horizon log returns
for h in (3, 5, 10):
    price_df[f'Ret_{h}d'] = np.log(price_df['Price'] / price_df['Price'].shift(h))

# ATR(14)
hl = np.log((price_df['High'] / price_df['Low']).clip(lower=1e-6))
hc = np.log((price_df['High'] / price_df['Price'].shift(1)).clip(lower=1e-6)).abs()
lc = np.log((price_df['Low']  / price_df['Price'].shift(1)).clip(lower=1e-6)).abs()
price_df['ATR14'] = pd.concat([hl, hc, lc], axis=1).max(axis=1).rolling(14).mean()

# Stochastic K(14), Williams %R(14), volatility-of-volatility
low14  = price_df['Low' ].rolling(14).min()
high14 = price_df['High'].rolling(14).max()
price_df['StochK14']    = (price_df['Price'] - low14) / (high14 - low14).replace(0, np.nan)
price_df['WilliamsR14'] = -100 * (high14 - price_df['Price']) / (high14 - low14).replace(0, np.nan)
price_df['VolOfVol']    = price_df['Volatility'].rolling(10).std()

# Calendar (sin/cos of day-of-week)
dow = price_df.index.dayofweek
price_df['DOW_sin'] = np.sin(2*np.pi*dow/5)
price_df['DOW_cos'] = np.cos(2*np.pi*dow/5)

# Decayed sentiment with strict no-look-ahead
HALF_LIFE_DAYS = 3
decay_rate     = np.log(2) / HALF_LIFE_DAYS
sentiment_dates_sorted = sorted(sentiment_map.keys())

def decayed_sentiment(date):
    '''Most-recent sentiment on or before (date - 1), exponentially decayed.'''
    query = (date - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
    best = None
    for sd in sentiment_dates_sorted:
        if sd <= query: best = sd
        else: break
    if best is None:
        return 0.0
    days_since = (pd.to_datetime(query) - pd.to_datetime(best)).days
    return sentiment_map[best] * np.exp(-decay_rate * days_since)

price_df['Sentiment']    = [decayed_sentiment(d) for d in price_df.index]
price_df['Sent_mean_3d'] = price_df['Sentiment'].rolling(3).mean()
price_df['Sent_std_7d']  = price_df['Sentiment'].rolling(7).std()

# Target: tomorrow's direction
price_df['Target'] = (price_df['Price'].shift(-1) > price_df['Price']).astype(int)
price_df.replace([np.inf, -np.inf], np.nan, inplace=True)

FEATURES = [
    'Price', 'Return', 'Volatility', 'RSI', 'MACD', 'BB_pctB', 'Vol_chg', 'Sentiment',
    'Ret_3d', 'Ret_5d', 'Ret_10d', 'ATR14', 'StochK14', 'WilliamsR14', 'VolOfVol',
    'DOW_sin', 'DOW_cos', 'Sent_mean_3d', 'Sent_std_7d',
]
price_df = price_df.dropna(subset=FEATURES + ['Target'])
print(f'Clean rows: {len(price_df)}   Up%: {price_df["Target"].mean():.2%}   |F| = {len(FEATURES)}')


### 3.5 Sequence construction and time-ordered splits

Each example is a 30-day window of features, news embeddings, and mask, ending on day $t$. The label is direction on $t+1$.

Split protocol:
- **Time-ordered.** Train = earliest 70%, validation = next 15%, test = most recent 15%.
- **Sequence-level cutoffs by target date.** A sequence belongs to a split based on its target date, not on whether any of its 30 lookback days is in the split.
- **No shuffling.** DataLoader uses `shuffle=False`.

Explicit assertions verify that splits are disjoint and cover the full sequence set.


In [ ]:
def create_sequences(df, feature_cols, lookback, news_embeddings):
    '''Build sequences. The target date is the LAST day of each window;
    the label is the direction on that day (which is shift(-1) of the label
    series, i.e. tomorrow's direction relative to that day).'''
    feats   = df[feature_cols].values.astype(np.float32)
    targets = df['Target'].values
    dates   = df.index.strftime('%Y-%m-%d').tolist()
    Xn, Xt, Xm, Y, D = [], [], [], [], []
    for i in range(len(df) - lookback):
        Xn.append(feats[i:i + lookback])
        # Target uses the LAST day of window: label is (tomorrow > today) for that day
        Y.append(targets[i + lookback - 1])
        D.append(dates[i + lookback - 1])
        # News embeddings: lagged by one CALENDAR day; zero on news-absent days
        ts, ms = [], []
        for d in dates[i:i + lookback]:
            lag = (pd.to_datetime(d) - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
            if news_embeddings and lag in news_embeddings:
                ts.append(news_embeddings[lag].numpy()); ms.append(1.0)
            else:
                ts.append(np.zeros(EMB_DIM, dtype=np.float32)); ms.append(0.0)
        Xt.append(ts); Xm.append(ms)
    return (np.asarray(Xn, dtype=np.float32),
            np.asarray(Xt, dtype=np.float32),
            np.asarray(Xm, dtype=np.float32),
            np.asarray(Y,  dtype=np.int64), D)

Xn_all, Xt_all, Xm_all, y_all, dates_all = create_sequences(
    price_df, FEATURES, LOOKBACK, news_emb)

# Time-ordered split, decided by sequence target date
n         = len(price_df)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)
train_cutoff = price_df.iloc[:train_end].index.max().strftime('%Y-%m-%d')
val_cutoff   = price_df.iloc[:val_end  ].index.max().strftime('%Y-%m-%d')

tr_idx = np.array([i for i, d in enumerate(dates_all) if d <= train_cutoff])
vl_idx = np.array([i for i, d in enumerate(dates_all) if train_cutoff < d <= val_cutoff])
te_idx = np.array([i for i, d in enumerate(dates_all) if d > val_cutoff])

# ── No-leakage assertions ──
assert len(set(tr_idx) & set(vl_idx)) == 0, 'train/val overlap'
assert len(set(vl_idx) & set(te_idx)) == 0, 'val/test overlap'
assert len(set(tr_idx) & set(te_idx)) == 0, 'train/test overlap'
assert len(tr_idx) + len(vl_idx) + len(te_idx) == len(dates_all), 'split coverage error'

# Test target dates must be strictly after all train target dates
test_dates  = [dates_all[i] for i in te_idx]
train_dates = [dates_all[i] for i in tr_idx]
assert min(test_dates) > max(train_dates), 'time order violated'

# Also: the test_dates we keep for backtesting later
test_dates_arr = np.array(test_dates)

news_frac_tr = Xm_all[tr_idx].mean()
print(f'Sequences shape (Xn, Xt, Xm): {Xn_all.shape}, {Xt_all.shape}, {Xm_all.shape}')
print(f'Split sizes  train={len(tr_idx)}  val={len(vl_idx)}  test={len(te_idx)}')
print(f'Up%          train={y_all[tr_idx].mean():.2%}  val={y_all[vl_idx].mean():.2%}  test={y_all[te_idx].mean():.2%}')
print(f'News-day %   train={news_frac_tr:.2%}')
print(f'Test period  {min(test_dates)} → {max(test_dates)}')


---

## 4. Building Blocks

### 4.1 Causal self-attention

Standard pre-LayerNorm transformer encoder block with a strict causal mask. The mask is applied to the (B, H, T, T) score matrix before softmax, not to the input embeddings. Setting `score[i, j] = -inf` for `j > i` makes `softmax(score)[i, j] = 0` exactly, so future values contribute nothing to position $i$'s output. Pre-LayerNorm (rather than post-LayerNorm) gives more stable gradients without learning-rate warmup hacks (Xiong et al. 2020).


In [ ]:
def _make_head(in_dim, hidden, dropout):
    '''Two-layer MLP classifier head.'''
    return nn.Sequential(
        nn.LayerNorm(in_dim), nn.Dropout(dropout),
        nn.Linear(in_dim, hidden), nn.GELU(),
        nn.LayerNorm(hidden), nn.Dropout(dropout),
        nn.Linear(hidden, 1),
    )


class CausalSelfAttention(nn.Module):
    '''Pre-norm transformer encoder with strict causal masking.

    Mask is applied to attention SCORES (not embeddings). Position i can
    attend only to j <= i. The mask uses -inf so softmax gives exactly 0
    weight to future positions.
    '''
    def __init__(self, d_model, num_heads=1, dropout=0.1, ff_mult=2, max_len=64):
        super().__init__()
        assert d_model % num_heads == 0
        self.h, self.dh = num_heads, d_model // num_heads
        self.d_model = d_model
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_mult * d_model), nn.GELU(),
            nn.Linear(ff_mult * d_model, d_model),
        )
        self.drop = nn.Dropout(dropout)
        self.pos_emb = nn.Parameter(torch.zeros(max_len, d_model))
        nn.init.normal_(self.pos_emb, std=0.02)

    def forward(self, x):
        B, T, D = x.shape
        x = x + self.pos_emb[:T].unsqueeze(0)
        h = self.ln1(x)
        qkv = self.qkv(h).view(B, T, 3, self.h, self.dh)
        q, k, v = qkv[..., 0, :, :], qkv[..., 1, :, :], qkv[..., 2, :, :]
        q = q.transpose(1, 2); k = k.transpose(1, 2); v = v.transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / (self.dh ** 0.5)
        # ── CAUSAL MASK ON SCORES ──
        causal_mask = torch.triu(
            torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(causal_mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.drop(attn)
        ctx  = (attn @ v).transpose(1, 2).contiguous().view(B, T, D)
        x = x + self.drop(self.out(ctx))
        x = x + self.drop(self.ffn(self.ln2(x)))
        return x


def _verify_causal_attention():
    '''Quick test: perturbing position j should not change output positions i < j.'''
    torch.manual_seed(0)
    layer = CausalSelfAttention(d_model=16, num_heads=2, dropout=0.0).eval()
    x = torch.randn(1, 8, 16)
    with torch.no_grad():
        out_base = layer(x)
        x_pert = x.clone()
        x_pert[:, 5:, :] += 100.0
        out_pert = layer(x_pert)
    diff = (out_base - out_pert).abs().max(dim=-1).values[0].cpu().numpy()
    for i, d in enumerate(diff):
        tag = '✓ no leak' if (i < 5 and d < 1e-5) else '✓ sees change'
        print(f'  position {i}: diff={d:.2e}  {tag}')
    assert all(diff[i] < 1e-5 for i in range(5)), 'CAUSAL ATTENTION LEAKS'
    print('  PASS')

_verify_causal_attention()


### 4.2 Poincaré ball operations

Numerical safety:
- All points are clipped to $\|\mathbf{x}\| \le (1 - 10^{-3})/\sqrt{c}$ to avoid the boundary singularity.
- The squared distance avoids $\sqrt{\cdot}$ near zero, which would produce unbounded gradients when two points are close.


In [ ]:
def project_to_ball(x, c):
    '''Clip ||x|| to slightly inside the boundary 1/sqrt(c).'''
    n  = torch.norm(x, dim=-1, keepdim=True).clamp(min=EPS)
    mn = MAX_NORM / (c ** 0.5)
    return x * torch.where(n > mn, mn / n, torch.ones_like(n))

def mobius_add(x, y, c):
    '''Möbius addition on the Poincaré ball of curvature c.'''
    x2 = (x*x).sum(-1, keepdim=True).clamp(min=0)
    y2 = (y*y).sum(-1, keepdim=True).clamp(min=0)
    xy = (x*y).sum(-1, keepdim=True)
    num = (1 + 2*c*xy + c*y2)*x + (1 - c*x2)*y
    den = (1 + 2*c*xy + c*c*x2*y2).clamp(min=EPS)
    return num / den

def exp_map_zero(v, c):
    '''Exp map at the origin: lifts v in tangent space to the ball.'''
    vn = torch.norm(v, dim=-1, keepdim=True).clamp(min=EPS)
    return project_to_ball(torch.tanh((c ** 0.5) * vn) * v / ((c ** 0.5) * vn), c)

def poincare_dist_sq(x, y, c):
    '''Squared Poincaré distance — avoids sqrt near zero.'''
    diff = mobius_add(-x, y, c)
    arg  = ((c ** 0.5) * torch.norm(diff, dim=-1).clamp(min=EPS)).clamp(max=1.0 - EPS)
    return (4.0 / c) * torch.atanh(arg) ** 2


### 4.3 Classical baselines

Three non-deep baselines that are standard reviewer-asks. The features going into them are the same 19 per day, but flattened into a single feature vector per example.

Each baseline uses a different summarisation of the 30-day window:

- **Logistic regression** — last-day features only, plus day-30 / day-1 deltas
- **XGBoost** — last day + window means + window stds (richer feature set, gradient-boosted)
- **ARIMA-X** — autoregressive on log return, sentiment as exogenous regressor

These are simple, fast, and serve as the floor. If a deep model can't beat XGBoost, we should know about it.


In [ ]:
def _flatten_features_for_classical(Xn, Xm):
    '''Build a tabular feature vector per example for non-deep baselines.
    Use last-day features + window means + window stds + news density.'''
    last_day = Xn[:, -1, :]                       # (N, F)
    win_mean = Xn.mean(axis=1)                    # (N, F)
    win_std  = Xn.std(axis=1)                     # (N, F)
    delta    = Xn[:, -1, :] - Xn[:, 0, :]         # (N, F)
    news_density = Xm.mean(axis=1, keepdims=True) # (N, 1)
    return np.concatenate([last_day, win_mean, win_std, delta, news_density], axis=1)


def fit_predict_logistic(Xn_tr, Xm_tr, y_tr, Xn_vl, Xm_vl, Xn_te, Xm_te, seed):
    '''Logistic regression on flattened features.'''
    Xtr = _flatten_features_for_classical(Xn_tr, Xm_tr)
    Xvl = _flatten_features_for_classical(Xn_vl, Xm_vl)
    Xte = _flatten_features_for_classical(Xn_te, Xm_te)
    sc  = StandardScaler().fit(Xtr)
    Xtr_s = sc.transform(Xtr); Xvl_s = sc.transform(Xvl); Xte_s = sc.transform(Xte)
    clf = LogisticRegression(max_iter=2000, C=1.0, random_state=seed)
    clf.fit(Xtr_s, y_tr)
    return clf.predict_proba(Xvl_s)[:, 1], clf.predict_proba(Xte_s)[:, 1]


def fit_predict_xgb(Xn_tr, Xm_tr, y_tr, Xn_vl, Xm_vl, Xn_te, Xm_te, seed):
    '''XGBoost on flattened features.'''
    Xtr = _flatten_features_for_classical(Xn_tr, Xm_tr)
    Xvl = _flatten_features_for_classical(Xn_vl, Xm_vl)
    Xte = _flatten_features_for_classical(Xn_te, Xm_te)
    clf = xgb.XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=seed, eval_metric='logloss',
        use_label_encoder=False, n_jobs=1)
    clf.fit(Xtr, y_tr)
    return clf.predict_proba(Xvl)[:, 1], clf.predict_proba(Xte)[:, 1]


def fit_predict_arimax(price_df, dates_tr, dates_vl, dates_te, seed):
    '''ARIMA(1,0,1) on log return with sentiment as exogenous variable.

    We fit on training dates, then forecast one-step-ahead probabilities
    by evaluating P(return > 0) under the predicted distribution at each
    val/test date. ARIMA does not have native classification probabilities;
    we use the predicted mean / std to compute P(return > 0).
    '''
    # Build a clean univariate series: log returns + sentiment exog
    ret = price_df['Return'].values
    sent = price_df['Sentiment'].values
    dates = price_df.index.strftime('%Y-%m-%d').tolist()
    date_to_idx = {d: i for i, d in enumerate(dates)}

    tr_end_idx = max(date_to_idx[d] for d in dates_tr if d in date_to_idx)
    # Fit ARIMAX on [start, tr_end_idx]
    try:
        model = ARIMA(ret[:tr_end_idx + 1],
                      exog=sent[:tr_end_idx + 1],
                      order=(1, 0, 1)).fit()
    except Exception:
        # Fallback if fit fails: predict majority class
        return (np.full(len(dates_vl), 0.5),
                np.full(len(dates_te), 0.5))

    # Walk-forward one-step predictions for val and test
    def predict_dates(target_dates):
        probs = []
        for td in target_dates:
            if td not in date_to_idx:
                probs.append(0.5); continue
            t_idx = date_to_idx[td]
            # Forecast using model fit on training; updates are not refit
            # to avoid expensive online refitting (this is standard practice)
            try:
                fc = model.apply(ret[:t_idx], exog=sent[:t_idx]).forecast(
                    steps=1, exog=sent[t_idx:t_idx+1].reshape(-1, 1))
                mu  = fc.iloc[0] if hasattr(fc, 'iloc') else fc[0]
                # Use predicted mean against zero; sigma approximated from residual std
                sigma = max(np.std(ret[max(0, t_idx-100):t_idx]) or 1e-3, 1e-3)
                p_up = 1.0 - stats.norm.cdf(0.0, loc=mu, scale=sigma)
                probs.append(float(p_up))
            except Exception:
                probs.append(0.5)
        return np.array(probs)

    return predict_dates(dates_vl), predict_dates(dates_te)


### 4.4 Deep baselines

Four deep models that share the same `CausalSelfAttention` block as CD-GSHA. This means any performance gap between baselines and CD-GSHA reflects only architectural choices in the news pathway (graph structure, geometry), not differences in the attention mechanism.

| Baseline | News access | Mechanism |
|---|---|---|
| LSTM-Base | None | Pure LSTM, last hidden state to head |
| LSTM-Attn | None | LSTM + causal self-attention over time |
| Cross-Attn | Yes (raw) | Causal self-attention on news, then price→news cross-attention |
| Dual-Attn | Yes (raw) | Parallel causal self-attention on each modality, last-step concatenation |


In [ ]:
class LSTMBase(nn.Module):
    '''Vanilla LSTM, no attention, no news.'''
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=0.2,
                 noise_std=0.0, **_):
        super().__init__()
        self.noise_std = noise_std
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.head = _make_head(hidden_dim, hidden_dim, dropout)
    def forward(self, xn, xt, xm):
        if self.training and self.noise_std > 0:
            xn = xn + torch.randn_like(xn) * self.noise_std
        h, _ = self.lstm(xn)
        return self.head(h[:, -1, :]).squeeze(-1)


class LSTMAttn(nn.Module):
    '''LSTM + causal self-attention. No news.'''
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=0.2,
                 num_heads=4, noise_std=0.0, **_):
        super().__init__()
        self.noise_std = noise_std
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.causal_attn = CausalSelfAttention(hidden_dim, num_heads=num_heads, dropout=dropout)
        self.head = _make_head(hidden_dim, hidden_dim, dropout)
    def forward(self, xn, xt, xm):
        if self.training and self.noise_std > 0:
            xn = xn + torch.randn_like(xn) * self.noise_std
        h, _ = self.lstm(xn)
        h    = self.causal_attn(h)
        return self.head(h[:, -1, :]).squeeze(-1)


class CrossAttn(nn.Module):
    '''News causally self-attended, then last-step price LSTM cross-attends.'''
    def __init__(self, input_dim, text_dim=EMB_DIM, hidden_dim=128, num_layers=2,
                 dropout=0.2, num_heads=4, noise_std=0.0, **_):
        super().__init__()
        self.noise_std = noise_std
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.text_proj = nn.Linear(text_dim, hidden_dim)
        self.text_causal_attn = CausalSelfAttention(hidden_dim, num_heads=num_heads, dropout=dropout)
        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)
        self.head   = _make_head(hidden_dim * 2, hidden_dim, dropout)
    def forward(self, xn, xt, xm):
        if self.training and self.noise_std > 0:
            xn = xn + torch.randn_like(xn) * self.noise_std
        h, _      = self.lstm(xn)
        price_ctx = h[:, -1, :]
        text      = self.text_proj(xt) * xm.unsqueeze(-1)
        text      = self.text_causal_attn(text)
        Q = self.q_proj(price_ctx).unsqueeze(1)
        K, V = self.k_proj(text), self.v_proj(text)
        scores = torch.bmm(Q, K.transpose(1, 2)) / (K.size(-1) ** 0.5)
        scores = scores.masked_fill(xm.unsqueeze(1) == 0, -1e9)
        attn   = F.softmax(scores, dim=-1)
        news_ctx = torch.bmm(attn, V).squeeze(1)
        return self.head(torch.cat([price_ctx, news_ctx], dim=-1)).squeeze(-1)


class DualAttn(nn.Module):
    '''Parallel causal self-attention per modality, last-step concat.'''
    def __init__(self, input_dim, text_dim=EMB_DIM, hidden_dim=128, num_layers=2,
                 dropout=0.2, num_heads=4, noise_std=0.0, **_):
        super().__init__()
        self.noise_std = noise_std
        self.price_lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True,
                                  dropout=dropout if num_layers > 1 else 0)
        self.text_proj = nn.Linear(text_dim, hidden_dim)
        self.price_causal = CausalSelfAttention(hidden_dim, num_heads=num_heads, dropout=dropout)
        self.text_causal  = CausalSelfAttention(hidden_dim, num_heads=num_heads, dropout=dropout)
        self.head = _make_head(hidden_dim * 2, hidden_dim, dropout)
    def forward(self, xn, xt, xm):
        if self.training and self.noise_std > 0:
            xn = xn + torch.randn_like(xn) * self.noise_std
        h, _ = self.price_lstm(xn)
        text = self.text_proj(xt) * xm.unsqueeze(-1)
        h    = self.price_causal(h)
        text = self.text_causal(text)
        return self.head(torch.cat([h[:, -1, :], text[:, -1, :]], dim=-1)).squeeze(-1)


---

## 5. Graph-Spectral Methods

We adapt directed-graph spectral filtering from the GSP literature (Defferrard et al. 2016 for symmetric Chebyshev; Tong et al. 2020 for directed-graph spectral methods; Rey et al. 2025 for DAG convolutional networks) to the temporal-graph setting on a per-window news graph.

The architecture has four learned components built on top of a shared price encoder: a graph builder, a Chebyshev convolution, a hybrid hyperbolic-Euclidean attention, and a confidence-gated cross-modal fusion.

### 5.1 Causal directional graph builder

Implements equations from §2.1. The graph has three edge types — semantic (between news-present days, gated by cosine similarity), temporal-prior (always-on, exponentially decaying), and self-loops — combined with learnable weights. By construction the graph is lower-triangular, so spectral propagation cannot move information from future to past.


In [ ]:
class CausalDirectionalGraphBuilder(nn.Module):
    '''Build a causal directed news graph and its random-walk Laplacian.'''
    def __init__(self, threshold_init=0.5, gamma_init=0.1, self_loop_init=0.5):
        super().__init__()
        self.log_tau   = nn.Parameter(torch.tensor(threshold_init).log())
        self.log_gamma = nn.Parameter(torch.tensor(gamma_init).log())
        self.alpha     = nn.Parameter(torch.tensor(1.0))
        self.beta      = nn.Parameter(torch.tensor(0.3))
        delta_init = float(np.log(np.exp(self_loop_init) - 1.0))
        self.log_delta = nn.Parameter(torch.tensor(delta_init))

    def forward(self, x_text, x_mask):
        B, T, _ = x_text.shape
        device  = x_text.device

        # Causal semantic edges (j < i, both with news)
        tau = torch.sigmoid(self.log_tau)
        nx  = F.normalize(x_text, dim=-1)
        sim = torch.bmm(nx, nx.transpose(1, 2))
        A_sem = torch.sigmoid((sim - tau) * 10.0)
        m2 = x_mask.unsqueeze(2) * x_mask.unsqueeze(1)
        A_sem = A_sem * m2

        # Causal temporal-prior edges (j < i)
        idx  = torch.arange(T, device=device, dtype=torch.float32)
        diff = idx.unsqueeze(1) - idx.unsqueeze(0)
        gamma = torch.exp(self.log_gamma)
        A_tmp = torch.exp(-gamma * diff.clamp(min=0.0))
        A_tmp = A_tmp.unsqueeze(0).expand(B, -1, -1)

        # Strict lower-triangular off-diagonal mask
        strict_lower = torch.tril(
            torch.ones(T, T, device=device, dtype=torch.float32), diagonal=-1)
        A_off = (self.alpha.abs() * A_sem + self.beta.abs() * A_tmp) * strict_lower

        # Diagonal: learnable self-loops
        delta = F.softplus(self.log_delta)
        I = torch.eye(T, device=device).unsqueeze(0).expand(B, -1, -1)
        A_diag = delta * I

        # Combined: lower-triangular incl. diagonal
        A = A_off + A_diag

        # Random-walk normalisation on the directed graph
        deg_out = A.sum(-1, keepdim=True).clamp(min=EPS)
        P = A / deg_out
        L_hat = I - P
        return L_hat, A


def _verify_directional_graph():
    torch.manual_seed(0)
    builder = CausalDirectionalGraphBuilder().eval()
    B, T = 2, 8
    x_text = torch.randn(B, T, EMB_DIM); x_mask = torch.ones(B, T)
    with torch.no_grad():
        L_hat, A = builder(x_text, x_mask)
    upper_A = torch.triu(A[0], diagonal=1)
    upper_L = torch.triu(L_hat[0], diagonal=1)
    print(f'  max strict-upper(A)     = {upper_A.abs().max().item():.2e}')
    print(f'  max strict-upper(L_hat) = {upper_L.abs().max().item():.2e}')
    print(f'  diag(A) (= delta)       = {torch.diagonal(A[0]).mean().item():.4f}')
    assert upper_A.abs().max() < 1e-9 and upper_L.abs().max() < 1e-9
    print('  PASS: directional graph and Laplacian are causal.')

_verify_directional_graph()


### 5.2 Causal Chebyshev convolution

Standard Chebyshev recursion (§2.3) applied to the directional Laplacian. Because $\hat{\mathbf{L}}^{\text{dir}}$ is lower-triangular, every $T_k(\hat{\mathbf{L}}^{\text{dir}})$ is lower-triangular by induction, so the convolution inherits causality from the graph topology.

A residual skip $\eta_r \mathbf{E}\mathbf{W}_r$ with fixed $\eta_r = 0.3$ lets the raw embedding signal bypass the graph if the graph is uninformative on a given window.


In [ ]:
class CausalChebyshevConv(nn.Module):
    def __init__(self, in_f, out_f, K=3, residual_scale=0.3):
        super().__init__()
        self.K = K
        self.W = nn.ParameterList([
            nn.Parameter(torch.randn(in_f, out_f) * 0.01) for _ in range(K)
        ])
        self.W_res = nn.Linear(in_f, out_f, bias=False)
        self.eta_r = residual_scale
        self.ln = nn.LayerNorm(out_f)

    def forward(self, x, L):
        '''x: (B, T, in_f); L: (B, T, T) lower-triangular Laplacian.'''
        Tp = x
        Tc = torch.bmm(L, x)
        coeffs = [Tp, Tc] if self.K >= 2 else [Tp]
        for k in range(2, self.K):
            Tn = 2 * torch.bmm(L, Tc) - Tp
            Tp, Tc = Tc, Tn
            coeffs.append(Tn)
        spec = sum(coeffs[k] @ self.W[k] for k in range(self.K))
        return self.ln(F.gelu(spec)) + self.eta_r * self.W_res(x)


def _verify_causal_chebyshev():
    '''Perturbing news at row j must not change output rows i < j.'''
    torch.manual_seed(0)
    builder = CausalDirectionalGraphBuilder().eval()
    conv    = CausalChebyshevConv(in_f=EMB_DIM, out_f=32, K=3).eval()
    B, T = 1, 8
    x      = torch.randn(B, T, EMB_DIM)
    x_mask = torch.ones(B, T)
    with torch.no_grad():
        L, _ = builder(x, x_mask)
        out_base = conv(x, L)
        j = 5
        x_pert = x.clone(); x_pert[:, j, :] += 100.0
        L_pert, _ = builder(x_pert, x_mask)
        out_pert  = conv(x_pert, L_pert)
    diff = (out_base - out_pert).abs().max(dim=-1).values[0].cpu().numpy()
    for i, d in enumerate(diff):
        tag = '✓ causal' if (i < j and d < 1e-4) else '✓ sees change'
        print(f'  output row {i}: diff={d:.2e}  {tag}')
    assert all(diff[i] < 1e-4 for i in range(j))
    print('  PASS: causality preserved end-to-end.')

_verify_causal_chebyshev()


### 5.3 Hybrid hyperbolic-Euclidean attention

Implements the score in §2.5. Single head ($H=1$); learnable mixing $\alpha, \beta$ via softplus; learnable shared curvature $c$ with floor at 0.1.


In [ ]:
class HybridHyperbolicAttention(nn.Module):
    def __init__(self, query_dim, key_dim, hyp_dim, num_heads=1, curvature_init=0.5):
        super().__init__()
        assert hyp_dim % num_heads == 0
        self.h, self.dh = num_heads, hyp_dim // num_heads
        self.q_proj = nn.Linear(query_dim, hyp_dim)
        self.k_proj = nn.Linear(key_dim, hyp_dim)
        self.v_proj = nn.Linear(key_dim, hyp_dim)
        self.out    = nn.Linear(hyp_dim, key_dim)
        self.log_alpha = nn.Parameter(torch.zeros(num_heads))
        self.log_beta  = nn.Parameter(torch.full((num_heads,), -2.3))
        theta_init = float(np.log(np.exp(curvature_init - 0.1) - 1.0))
        self.log_curvature = nn.Parameter(torch.tensor(theta_init))
        self.scale = self.dh ** -0.5

    def get_alpha(self):     return F.softplus(self.log_alpha)
    def get_beta(self):      return F.softplus(self.log_beta)
    def get_curvature(self): return F.softplus(self.log_curvature) + 0.1

    def forward(self, query, keys):
        '''query: (B, query_dim); keys: (B, T, key_dim).'''
        B, T, _ = keys.shape
        q = self.q_proj(query).view(B, self.h, self.dh)
        k = self.k_proj(keys ).view(B, T, self.h, self.dh)
        v = self.v_proj(keys ).view(B, T, self.h, self.dh)
        eucl = (q.unsqueeze(1) * k).sum(-1) * self.scale
        c = self.get_curvature()
        q_ball = exp_map_zero(q.reshape(-1, self.dh), c).view(B, self.h, self.dh)
        k_ball = exp_map_zero(k.reshape(-1, self.dh), c).view(B, T, self.h, self.dh)
        q_exp  = q_ball.unsqueeze(1).expand(-1, T, -1, -1)
        d_sq   = poincare_dist_sq(q_exp, k_ball, c)
        alpha = self.get_alpha().view(1, 1, self.h)
        beta  = self.get_beta ().view(1, 1, self.h)
        scores = alpha * eucl - beta * c * d_sq
        attn   = F.softmax(scores, dim=1)
        ctx = (attn.unsqueeze(-1) * v).sum(dim=1).reshape(B, self.h * self.dh)
        return self.out(ctx), attn


### 5.4 Full CD-GSHA model

Ties everything together:

1. Price branch: LSTM → causal self-attention → last-step projected
2. News branch: directional graph builder → causal Chebyshev (no separate causal mask needed — the graph is causal)
3. Cross-modal: hybrid hyperbolic-Euclidean attention with price as query
4. Fusion: confidence gate conditioned on news density
5. Sentiment side-channel: small LSTM over sentiment column
6. Classifier: MLP over concatenated [fused; sentiment]


In [ ]:
class CDGSHA(nn.Module):
    def __init__(self, input_dim, text_dim=EMB_DIM,
                 hidden_dim=128, num_layers=2, dropout=0.2,
                 gnn_dim=64, hyp_dim=32, cheb_K=3, num_heads=4,
                 noise_std=0.0, asymmetric_bias=0.0, **_):
        super().__init__()
        self.noise_std = noise_std
        gsha_heads = 1   # ablation-driven design choice

        # Price branch
        self.price_lstm   = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True,
                                    dropout=dropout if num_layers > 1 else 0)
        self.price_causal = CausalSelfAttention(hidden_dim, num_heads=gsha_heads, dropout=dropout)
        self.price_proj   = nn.Linear(hidden_dim, gnn_dim)

        # News branch
        self.graph_builder = CausalDirectionalGraphBuilder()
        self.cheb_conv     = CausalChebyshevConv(text_dim, gnn_dim, K=cheb_K)

        # Cross-modal hybrid attention
        self.hyp_attn = HybridHyperbolicAttention(gnn_dim, gnn_dim, hyp_dim,
                                                  num_heads=gsha_heads)

        # Confidence-gated fusion
        self.rho_gate = nn.Sequential(
            nn.Linear(gnn_dim*2 + 1, gnn_dim), nn.GELU(),
            nn.Linear(gnn_dim, gnn_dim), nn.Sigmoid())

        # Sentiment side-channel
        self.sent_lstm = nn.LSTM(1, max(hidden_dim // 4, 4), 1, batch_first=True)
        self.sent_proj = nn.Linear(max(hidden_dim // 4, 4), gnn_dim)

        self.classifier = _make_head(gnn_dim * 2, gnn_dim, dropout)
        self.bias = asymmetric_bias

    def forward(self, xn, xt, xm):
        if self.training and self.noise_std > 0:
            xn = xn + torch.randn_like(xn) * self.noise_std
        po, _ = self.price_lstm(xn)
        po    = self.price_causal(po)
        pctx  = self.price_proj(po[:, -1, :])
        L_dir, _ = self.graph_builder(xt, xm)
        Z = self.cheb_conv(xt, L_dir)
        c_hyp, _ = self.hyp_attn(pctx, Z)
        nd    = xm.mean(dim=1, keepdim=True)
        rho   = self.rho_gate(torch.cat([pctx, c_hyp, nd], dim=-1))
        fused = rho * c_hyp + (1 - rho) * pctx
        so, _ = self.sent_lstm(xn[:, :, -1:].contiguous())
        sctx  = self.sent_proj(so[:, -1, :])
        return self.classifier(torch.cat([fused, sctx], dim=-1)).squeeze(-1) + self.bias


def _verify_cd_gsha_causality():
    torch.manual_seed(0)
    model = CDGSHA(input_dim=19, hidden_dim=32, num_layers=1, gnn_dim=16,
                   hyp_dim=16, cheb_K=2).eval()
    B, T = 1, 10
    xt = torch.randn(B, T, 16); xm = torch.ones(B, T)
    with torch.no_grad():
        L0, _ = model.graph_builder(xt, xm)
        Z0    = model.cheb_conv(xt, L0)
        j = 6
        xt_p = xt.clone(); xt_p[:, j, :] += 100.0
        L1, _ = model.graph_builder(xt_p, xm)
        Z1    = model.cheb_conv(xt_p, L1)
    diff = (Z0 - Z1).abs().max(dim=-1).values[0].cpu().numpy()
    for i, d in enumerate(diff):
        tag = '✓ causal' if (i < j and d < 1e-4) else '✓ sees change'
        print(f'  Z row {i}: diff={d:.2e}  {tag}')
    assert all(diff[i] < 1e-4 for i in range(j))
    print('  PASS: end-to-end CD-GSHA news pathway is causal.')

_verify_cd_gsha_causality()


### 5.5 Symmetric-graph variant (a baseline ablation)

To isolate the contribution of *directionality* we include a variant that uses the standard symmetric Laplacian $\mathbf{D}^{-1/2}\mathbf{A}\mathbf{D}^{-1/2}$ on a symmetric news graph. This is the architecture from Defferrard et al. 2016 with our learnable graph priors but without causality. If directionality matters, this variant should underperform; if not, the simpler symmetric architecture is preferable.


In [ ]:
class _SymmetricGraphBuilder(nn.Module):
    def __init__(self, threshold_init=0.5, gamma_init=0.1):
        super().__init__()
        self.log_tau   = nn.Parameter(torch.tensor(threshold_init).log())
        self.log_gamma = nn.Parameter(torch.tensor(gamma_init).log())
        self.alpha     = nn.Parameter(torch.tensor(1.0))
        self.beta      = nn.Parameter(torch.tensor(0.3))
    def forward(self, x_text, x_mask):
        B, T, _ = x_text.shape
        device  = x_text.device
        tau   = torch.sigmoid(self.log_tau)
        nx    = F.normalize(x_text, dim=-1)
        sim   = torch.bmm(nx, nx.transpose(1, 2))
        A_sem = torch.sigmoid((sim - tau) * 10.0)
        m2    = x_mask.unsqueeze(2) * x_mask.unsqueeze(1)
        A_sem = A_sem * m2
        idx   = torch.arange(T, device=device, dtype=torch.float32)
        dist  = (idx.unsqueeze(0) - idx.unsqueeze(1)).abs()
        gamma = torch.exp(self.log_gamma)
        A_tmp = torch.exp(-gamma * dist).unsqueeze(0).expand(B, -1, -1)
        A = self.alpha.abs() * A_sem + self.beta.abs() * A_tmp
        I = torch.eye(T, device=device).unsqueeze(0).expand(B, -1, -1)
        A = A * (1 - I)
        deg = A.sum(-1).clamp(min=EPS)
        Di  = (deg ** -0.5).unsqueeze(-1)
        L_sym = Di * A * Di.transpose(1, 2)
        return L_sym, A


class GSHA_Symmetric(CDGSHA):
    '''CD-GSHA with the directional graph replaced by a symmetric one.
    Re-introduces the future-to-past leakage that CD-GSHA prevents.
    Note: in a symmetric-graph version we add a news-side causal self-attn
    AFTER the spectral conv to enforce causality, since the graph itself
    no longer enforces it.
    '''
    def __init__(self, *a, **kw):
        super().__init__(*a, **kw)
        self.graph_builder = _SymmetricGraphBuilder()
        self.news_causal = CausalSelfAttention(kw.get('gnn_dim', 64), num_heads=1,
                                               dropout=kw.get('dropout', 0.2))
    def forward(self, xn, xt, xm):
        if self.training and self.noise_std > 0:
            xn = xn + torch.randn_like(xn) * self.noise_std
        po, _ = self.price_lstm(xn)
        po    = self.price_causal(po)
        pctx  = self.price_proj(po[:, -1, :])
        L_sym, _ = self.graph_builder(xt, xm)
        Z = self.cheb_conv(xt, L_sym)
        Z = self.news_causal(Z)   # post-hoc causal mask, since L_sym isn't causal
        c_hyp, _ = self.hyp_attn(pctx, Z)
        nd    = xm.mean(dim=1, keepdim=True)
        rho   = self.rho_gate(torch.cat([pctx, c_hyp, nd], dim=-1))
        fused = rho * c_hyp + (1 - rho) * pctx
        so, _ = self.sent_lstm(xn[:, :, -1:].contiguous())
        sctx  = self.sent_proj(so[:, -1, :])
        return self.classifier(torch.cat([fused, sctx], dim=-1)).squeeze(-1) + self.bias


### 5.6 CD-GSHA ablations

Each ablation removes or modifies one component of CD-GSHA, holding everything else fixed:

| Variant | What changes |
|---|---|
| `CD-GSHA - temporal priors` | $\beta_g$ frozen at 0; only semantic edges remain |
| `CD-GSHA - graph` | $\hat{\mathbf{L}}^{\text{dir}} = 0$; ChebConv reduces to a per-day linear map |
| `CD-GSHA - hyperbolic` | $\beta = 0$ frozen; pure Euclidean attention |
| `CD-GSHA - euclidean` | $\alpha = 0$ frozen; pure hyperbolic attention |
| `CD-GSHA - price causal` | Replace price-side `CausalSelfAttention` with Identity |
| `CD-GSHA + multi-head` | Use 4 heads in hybrid attention instead of 1 |


In [ ]:
class CDGSHA_NoTemporal(CDGSHA):
    def __init__(self, *a, **kw):
        super().__init__(*a, **kw)
        with torch.no_grad():
            self.graph_builder.beta.data.fill_(0.0)
        self.graph_builder.beta.requires_grad_(False)


class CDGSHA_NoGraph(CDGSHA):
    def forward(self, xn, xt, xm):
        if self.training and self.noise_std > 0:
            xn = xn + torch.randn_like(xn) * self.noise_std
        po, _ = self.price_lstm(xn)
        po    = self.price_causal(po)
        pctx  = self.price_proj(po[:, -1, :])
        B, T, _ = xt.shape
        L_zero = torch.zeros(B, T, T, device=xt.device)
        Z = self.cheb_conv(xt, L_zero)
        c_hyp, _ = self.hyp_attn(pctx, Z)
        nd    = xm.mean(dim=1, keepdim=True)
        rho   = self.rho_gate(torch.cat([pctx, c_hyp, nd], dim=-1))
        fused = rho * c_hyp + (1 - rho) * pctx
        so, _ = self.sent_lstm(xn[:, :, -1:].contiguous())
        sctx  = self.sent_proj(so[:, -1, :])
        return self.classifier(torch.cat([fused, sctx], dim=-1)).squeeze(-1) + self.bias


class CDGSHA_NoHyperbolic(CDGSHA):
    def __init__(self, *a, **kw):
        super().__init__(*a, **kw)
        with torch.no_grad():
            self.hyp_attn.log_beta.data.fill_(-20.0)
        self.hyp_attn.log_beta.requires_grad_(False)


class CDGSHA_NoEuclidean(CDGSHA):
    def __init__(self, *a, **kw):
        super().__init__(*a, **kw)
        with torch.no_grad():
            self.hyp_attn.log_alpha.data.fill_(-20.0)
        self.hyp_attn.log_alpha.requires_grad_(False)


class CDGSHA_NoPriceCausal(CDGSHA):
    def __init__(self, *a, **kw):
        super().__init__(*a, **kw)
        self.price_causal = nn.Identity()


class CDGSHA_MultiHead(CDGSHA):
    '''Override the single-head choice to test 4 heads in hybrid attention.'''
    def __init__(self, input_dim, text_dim=EMB_DIM, hidden_dim=128, num_layers=2,
                 dropout=0.2, gnn_dim=64, hyp_dim=32, cheb_K=3, num_heads=4,
                 noise_std=0.0, asymmetric_bias=0.0, **_):
        # Skip CDGSHA.__init__ and set up with multi-head
        nn.Module.__init__(self)
        self.noise_std = noise_std
        gsha_heads = 4
        self.price_lstm   = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True,
                                    dropout=dropout if num_layers > 1 else 0)
        self.price_causal = CausalSelfAttention(hidden_dim, num_heads=gsha_heads, dropout=dropout)
        self.price_proj   = nn.Linear(hidden_dim, gnn_dim)
        self.graph_builder = CausalDirectionalGraphBuilder()
        self.cheb_conv     = CausalChebyshevConv(text_dim, gnn_dim, K=cheb_K)
        self.hyp_attn = HybridHyperbolicAttention(gnn_dim, gnn_dim, hyp_dim,
                                                  num_heads=gsha_heads)
        self.rho_gate = nn.Sequential(
            nn.Linear(gnn_dim*2 + 1, gnn_dim), nn.GELU(),
            nn.Linear(gnn_dim, gnn_dim), nn.Sigmoid())
        self.sent_lstm = nn.LSTM(1, max(hidden_dim // 4, 4), 1, batch_first=True)
        self.sent_proj = nn.Linear(max(hidden_dim // 4, 4), gnn_dim)
        self.classifier = _make_head(gnn_dim * 2, gnn_dim, dropout)
        self.bias = asymmetric_bias


---

## 6. Training Protocol

All deep models are trained with the same loss, optimizer, schedule, and ensemble protocol. Any performance difference therefore reflects only the architecture.

### 6.1 Loss

Class-balanced binary focal loss with label smoothing (Lin et al. 2017):

- Class weight $w_+ = (1 - \pi)/\pi$ from the train positive rate prevents the model from collapsing to predicting the majority class
- Focal $(1 - p_t)^\gamma$ with $\gamma = 2$ down-weights easy examples; gradient signal concentrates on hard examples near the boundary
- Label smoothing $\varepsilon = 0.05$ keeps $\sigma(z)$ off the asymptotic 0/1 ends

### 6.2 Threshold selection on validation only

Threshold selection is done on the *validation* ensemble probabilities. The test set is touched once with the val-chosen threshold.


In [ ]:
class FocalBCE(nn.Module):
    def __init__(self, gamma=2.0, label_smoothing=0.0, pos_weight=1.0):
        super().__init__()
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        self.register_buffer('pos_weight', torch.tensor(float(pos_weight)))
    def forward(self, logit, y):
        if self.label_smoothing > 0:
            y = y * (1 - self.label_smoothing) + 0.5 * self.label_smoothing
        p  = torch.sigmoid(logit)
        pt = torch.where(y >= 0.5, p, 1 - p)
        w  = torch.where(y >= 0.5, self.pos_weight, torch.ones_like(self.pos_weight))
        return -(w * (1 - pt) ** self.gamma * torch.log(pt.clamp(min=EPS))).mean()


def best_threshold(y_true, y_prob, grid=None, min_class_frac=0.25):
    '''Macro-F1 max on grid, with diversity constraint to avoid degenerate
    thresholds that predict ~all one class.'''
    if grid is None:
        grid = np.linspace(0.40, 0.60, 21)
    best_t, best_macro = 0.5, -1.0
    for t in grid:
        pred = (y_prob > t).astype(int)
        pp = pred.mean()
        if pp < min_class_frac or pp > 1 - min_class_frac:
            continue
        macro = f1_score(y_true, pred, average='macro', zero_division=0)
        if macro > best_macro:
            best_macro, best_t = macro, t
    return float(best_t), float(best_macro)


def evaluate_classification(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob > threshold).astype(int)
    return dict(
        threshold     = threshold,
        acc           = accuracy_score(y_true, y_pred),
        balanced_acc  = balanced_accuracy_score(y_true, y_pred),
        f1            = f1_score(y_true, y_pred, average='macro', zero_division=0),
        mcc           = matthews_corrcoef(y_true, y_pred),
        prec_up       = precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        prec_down     = precision_score(y_true, y_pred, pos_label=0, zero_division=0),
        recall_up     = recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        recall_down   = recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        auc           = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else float('nan'),
        pred_pos_rate = float(y_pred.mean()),
    )


### 6.3 Training loop

- AdamW + OneCycleLR (warmup 20%, cosine anneal)
- Per-parameter LR group: hyperbolic-mixing parameters get a 5× LR multiplier so $\beta$ and $c$ actually move during training (without this, the softplus parametrisation makes their effective gradients much smaller than other params)
- Gradient clipping at $\|\nabla\| \le 1$
- Stochastic weight averaging (SWA) over the last 25% of epochs; SWA-vs-checkpoint choice judged on *validation* macro-F1
- StandardScaler is fit on training data only and applied to validation/test using train statistics


In [ ]:
def _make_dataset(xn, xt, xm, y):
    return TensorDataset(torch.from_numpy(xn), torch.from_numpy(xt),
                         torch.from_numpy(xm), torch.from_numpy(y.astype(np.float32)))

def _make_loader(xn, xt, xm, y, batch_size=32, shuffle=False):
    return DataLoader(_make_dataset(xn, xt, xm, y), batch_size=batch_size, shuffle=shuffle)


def _hyperbolic_param_names(model):
    '''Names of params that get the higher-LR group (only present in CD-GSHA).'''
    return [n for n, _ in model.named_parameters()
            if any(k in n for k in ('log_alpha', 'log_beta', 'log_curvature',
                                    'log_delta', 'log_tau', 'log_gamma'))]


def train_deep_model(model_class, params,
                     Xn_tr, Xt_tr, Xm_tr, y_tr,
                     Xn_vl, Xt_vl, Xm_vl, y_vl,
                     seed, verbose=False):
    '''Train one deep model. Returns (final_model, scaler, val_probs).'''
    set_seed(seed)
    F_ = Xn_tr.shape[-1]

    sc = StandardScaler().fit(Xn_tr.reshape(-1, F_))
    Xn_tr_s = sc.transform(Xn_tr.reshape(-1, F_)).reshape(Xn_tr.shape).astype(np.float32)
    Xn_vl_s = sc.transform(Xn_vl.reshape(-1, F_)).reshape(Xn_vl.shape).astype(np.float32)

    tr_loader = _make_loader(Xn_tr_s, Xt_tr, Xm_tr, y_tr,
                             batch_size=params['batch_size'], shuffle=params['shuffle'])
    vl_loader = _make_loader(Xn_vl_s, Xt_vl, Xm_vl, y_vl,
                             batch_size=params['batch_size'])

    model = model_class(
        input_dim       = F_,
        text_dim        = EMB_DIM,
        hidden_dim      = params['hidden_dim'],
        num_layers      = params['num_layers'],
        dropout         = params['dropout'],
        gnn_dim         = params['gnn_dim'],
        hyp_dim         = params['hyp_dim'],
        cheb_K          = params['cheb_K'],
        num_heads       = params['num_heads'],
        noise_std       = params['noise_std'],
        asymmetric_bias = params['asymmetric_bias'],
    ).to(DEVICE)

    pos_rate   = float(y_tr.mean())
    pos_weight = (1.0 - pos_rate) / max(pos_rate, 1e-6)

    hyp_names = set(_hyperbolic_param_names(model))
    high_lr   = [p for n, p in model.named_parameters() if n in hyp_names]
    base      = [p for n, p in model.named_parameters() if n not in hyp_names]
    lr_mult   = params.get('hyp_lr_mult', 5.0)

    if len(high_lr) > 0:
        param_groups = [
            {'params': base,    'lr': params['lr']},
            {'params': high_lr, 'lr': params['lr'] * lr_mult, 'weight_decay': 0.0},
        ]
        max_lrs = [params['lr'], params['lr'] * lr_mult]
    else:
        param_groups = [{'params': base, 'lr': params['lr']}]
        max_lrs = [params['lr']]

    optim = torch.optim.AdamW(param_groups, lr=params['lr'],
                              weight_decay=params['weight_decay'])
    total_steps = params['max_epochs'] * len(tr_loader)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        optim, max_lr=max_lrs, total_steps=total_steps,
        pct_start=0.2, div_factor=25.0, final_div_factor=1e4)
    bce = FocalBCE(gamma=params['focal_gamma'],
                   label_smoothing=params['label_smoothing'],
                   pos_weight=pos_weight).to(DEVICE)

    swa_model = AveragedModel(model)
    swa_start = int(params['max_epochs'] * 0.75)

    best_vl, best_state, no_improve = float('inf'), None, 0
    for epoch in range(params['max_epochs']):
        model.train()
        for xn, xt, xm, yb in tr_loader:
            xn, xt, xm, yb = [t.to(DEVICE) for t in (xn, xt, xm, yb)]
            optim.zero_grad()
            loss = bce(model(xn, xt, xm), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step(); sched.step()
        if epoch >= swa_start:
            swa_model.update_parameters(model)
        model.eval()
        vl = 0.0
        with torch.no_grad():
            for xn, xt, xm, yb in vl_loader:
                xn, xt, xm, yb = [t.to(DEVICE) for t in (xn, xt, xm, yb)]
                vl += bce(model(xn, xt, xm), yb).item()
        vl /= len(vl_loader)
        if vl < best_vl:
            best_vl, best_state, no_improve = vl, copy.deepcopy(model.state_dict()), 0
        else:
            no_improve += 1
            if no_improve >= params['patience']:
                break

    model.load_state_dict(best_state); model.eval(); swa_model.eval()
    p_best, p_swa = [], []
    with torch.no_grad():
        for xn, xt, xm, _ in vl_loader:
            xn, xt, xm = [t.to(DEVICE) for t in (xn, xt, xm)]
            p_best.append(torch.sigmoid(model    (xn, xt, xm)).cpu().numpy())
            p_swa .append(torch.sigmoid(swa_model(xn, xt, xm)).cpu().numpy())
    p_best = np.concatenate(p_best); p_swa = np.concatenate(p_swa)
    f1_best = f1_score(y_vl, (p_best > 0.5).astype(int), average='macro', zero_division=0)
    f1_swa  = f1_score(y_vl, (p_swa  > 0.5).astype(int), average='macro', zero_division=0)
    chosen, val_prob = (swa_model, p_swa) if f1_swa >= f1_best else (model, p_best)
    return chosen, sc, val_prob


def predict_deep(model, scaler, Xn, Xt, Xm, batch_size=32):
    model.eval()
    Xn_s = scaler.transform(Xn.reshape(-1, Xn.shape[-1])).reshape(Xn.shape).astype(np.float32)
    loader = _make_loader(Xn_s, Xt, Xm, np.zeros(len(Xn), dtype=np.int64), batch_size=batch_size)
    probs = []
    with torch.no_grad():
        for xn, xt, xm, _ in loader:
            xn, xt, xm = [t.to(DEVICE) for t in (xn, xt, xm)]
            probs.append(torch.sigmoid(model(xn, xt, xm)).cpu().numpy())
    return np.concatenate(probs)


### 6.4 Hyperparameters

Single source of truth for all training hyperparameters. Identical for every deep model.


In [ ]:
HPARAMS = dict(
    hidden_dim      = 128,
    num_layers      = 2,
    dropout         = 0.20,
    gnn_dim         = 64,
    hyp_dim         = 32,
    cheb_K          = 3,
    num_heads       = 4,            # baselines; CD-GSHA overrides to 1 internally
    focal_gamma     = 2.0,
    label_smoothing = 0.05,
    asymmetric_bias = 0.0,
    noise_std       = 0.02,
    lr              = 1e-3,
    weight_decay    = 1e-4,
    hyp_lr_mult     = 5.0,
    max_epochs      = 120,
    patience        = 20,
    batch_size      = 32,
    shuffle         = False,
    seeds_main      = list(range(20)),    # 20 seeds for headline comparison
    seeds_ablation  = list(range(10)),    # 10 seeds for ablations
)

print('HPARAMS:')
for k, v in HPARAMS.items():
    if isinstance(v, list) and len(v) > 4:
        print(f'  {k:18s}: list of {len(v)} ({v[:3]}...{v[-1]})')
    else:
        print(f'  {k:18s}: {v}')


### 6.5 Multi-seed runner

Trains a model class on multiple seeds, averages test probabilities across seeds, picks threshold on the validation ensemble. Returns metrics + raw test probabilities (for downstream backtesting and significance tests).


In [ ]:
def run_deep_experiment(model_class, name, seeds, verbose=True):
    '''Multi-seed deep training with ensembling. Returns dict with metrics
    + test_prob (ensemble) + per-seed probs (for significance tests).'''
    t0 = time.time()
    val_probs_all, test_probs_all = [], []
    for seed in seeds:
        model, scaler, p_vl = train_deep_model(
            model_class, HPARAMS,
            Xn_all[tr_idx], Xt_all[tr_idx], Xm_all[tr_idx], y_all[tr_idx],
            Xn_all[vl_idx], Xt_all[vl_idx], Xm_all[vl_idx], y_all[vl_idx],
            seed=seed, verbose=False)
        val_probs_all.append(p_vl)
        p_te = predict_deep(model, scaler, Xn_all[te_idx], Xt_all[te_idx], Xm_all[te_idx])
        test_probs_all.append(p_te)

    val_ensemble  = np.mean(val_probs_all,  axis=0)
    test_ensemble = np.mean(test_probs_all, axis=0)

    t_star, val_f1 = best_threshold(y_all[vl_idx], val_ensemble)
    res = evaluate_classification(y_all[te_idx], test_ensemble, threshold=t_star)
    res['val_f1']   = val_f1
    res['runtime']  = time.time() - t0
    res['n_seeds']  = len(seeds)
    res['test_prob']         = test_ensemble
    res['test_probs_per_seed'] = np.array(test_probs_all)
    res['threshold']         = t_star

    if verbose:
        print(f'  {name:24s}  thr={t_star:.3f}  val_F1={val_f1:.4f}  '
              f'test_F1={res["f1"]:.4f}  MCC={res["mcc"]:.4f}  '
              f'BalAcc={res["balanced_acc"]:.4f}  ({res["runtime"]/60:.1f} min)')
    return res


def run_classical_experiment(name, fit_fn, seeds, verbose=True):
    '''Multi-seed classical baseline runner. fit_fn signature varies; we
    handle the three cases inline.'''
    t0 = time.time()
    val_probs_all, test_probs_all = [], []
    for seed in seeds:
        if name == 'ARIMA-X':
            # ARIMA-X is deterministic; only need one fit
            if seed != seeds[0]:
                val_probs_all.append(val_probs_all[0])
                test_probs_all.append(test_probs_all[0])
                continue
            dates_tr = [dates_all[i] for i in tr_idx]
            dates_vl = [dates_all[i] for i in vl_idx]
            dates_te = [dates_all[i] for i in te_idx]
            p_vl, p_te = fit_fn(price_df, dates_tr, dates_vl, dates_te, seed)
        else:
            p_vl, p_te = fit_fn(
                Xn_all[tr_idx], Xm_all[tr_idx], y_all[tr_idx],
                Xn_all[vl_idx], Xm_all[vl_idx],
                Xn_all[te_idx], Xm_all[te_idx], seed)
        val_probs_all.append(p_vl)
        test_probs_all.append(p_te)

    val_ensemble  = np.mean(val_probs_all,  axis=0)
    test_ensemble = np.mean(test_probs_all, axis=0)
    t_star, val_f1 = best_threshold(y_all[vl_idx], val_ensemble)
    res = evaluate_classification(y_all[te_idx], test_ensemble, threshold=t_star)
    res['val_f1']  = val_f1
    res['runtime'] = time.time() - t0
    res['n_seeds'] = len(seeds)
    res['test_prob']           = test_ensemble
    res['test_probs_per_seed'] = np.array(test_probs_all)
    res['threshold']           = t_star
    if verbose:
        print(f'  {name:24s}  thr={t_star:.3f}  val_F1={val_f1:.4f}  '
              f'test_F1={res["f1"]:.4f}  MCC={res["mcc"]:.4f}  '
              f'BalAcc={res["balanced_acc"]:.4f}  ({res["runtime"]/60:.1f} min)')
    return res


---

## 7. Headline Experiments

We train all nine architectures with 20 seeds each. Test probabilities are averaged across seeds; the threshold is chosen on the validation ensemble.

### 7.1 Model registry


In [ ]:
DEEP_MODEL_CLASSES = {
    'LSTM-Base':       LSTMBase,
    'LSTM-Attn':       LSTMAttn,
    'Cross-Attn':      CrossAttn,
    'Dual-Attn':       DualAttn,
    'GSHA-Symmetric':  GSHA_Symmetric,
    'CD-GSHA':         CDGSHA,
}

CLASSICAL_MODELS = {
    'Logistic':  fit_predict_logistic,
    'XGBoost':   fit_predict_xgb,
    'ARIMA-X':   fit_predict_arimax,
}


### 7.2 Run headline comparison


In [ ]:
print('=' * 80)
print(f' HEADLINE EXPERIMENTS — {COMMODITY.upper()} — {len(HPARAMS["seeds_main"])} seeds')
print('=' * 80)

results = {}

print('\nClassical baselines:')
for name, fn in CLASSICAL_MODELS.items():
    seeds = HPARAMS['seeds_main'] if name != 'ARIMA-X' else [0]
    results[name] = run_classical_experiment(name, fn, seeds, verbose=True)

print('\nDeep models:')
for name, cls in DEEP_MODEL_CLASSES.items():
    results[name] = run_deep_experiment(cls, name, HPARAMS['seeds_main'], verbose=True)


### 7.3 Headline metrics table


In [ ]:
def _row(name, r):
    return {
        'Model':        name,
        'Accuracy':     r['acc'],
        'Balanced Acc': r['balanced_acc'],
        'F1':           r['f1'],
        'MCC':          r['mcc'],
        'AUC':          r['auc'],
        'Prec (Up)':    r['prec_up'],
        'Prec (Down)':  r['prec_down'],
        'Pred Up%':     r['pred_pos_rate'],
        'Threshold':    r['threshold'],
    }

results_df = pd.DataFrame([_row(n, r) for n, r in results.items()]).set_index('Model')
primary = ['Accuracy', 'Balanced Acc', 'F1', 'MCC', 'AUC']
detail  = ['Prec (Up)', 'Prec (Down)', 'Pred Up%', 'Threshold']

print('=' * 92)
print(f'  TEST RESULTS — PRIMARY METRICS — {COMMODITY.upper()}')
print('=' * 92)
print(results_df[primary].to_string(float_format=lambda v: f'{v:.4f}'))
print()
print('=' * 92)
print('  TEST RESULTS — DIAGNOSTIC DETAIL')
print('=' * 92)
print(results_df[detail].to_string(float_format=lambda v: f'{v:.4f}'))

print()
for col in ['F1', 'MCC', 'Balanced Acc', 'Accuracy', 'AUC']:
    winner = results_df[col].idxmax()
    print(f'Best {col:14s}: {winner:18s} ({results_df.loc[winner, col]:.4f})')


---

## 8. Statistical Significance

Two tests, both standard in the forecasting literature:

### 8.1 Diebold-Mariano test (forecast errors)

Compares two forecasts via the difference in their loss series. Tests $H_0$: equal expected loss. Implementation uses Newey-West HAC adjustment for autocorrelation in the loss differential. Reports the modified small-sample DM statistic of Harvey, Leybourne, and Newbold (1997).

### 8.2 Pesaran-Timmermann test (directional accuracy)

The DM test compares forecast errors but doesn't directly test directional accuracy. PT tests $H_0$: directional predictions are independent of actual direction (i.e., the directional hit rate is no better than expected by chance given the marginals).


In [ ]:
def diebold_mariano(loss1, loss2, h=1):
    '''Modified DM test (Harvey, Leybourne, Newbold 1997) on two loss series.
    Two-sided p-value. Returns (DM_stat, p_value, mean_diff).
    Negative DM_stat means model 1 has lower loss.'''
    d = np.asarray(loss1) - np.asarray(loss2)
    d = d[~np.isnan(d)]
    n = len(d)
    if n < 5:
        return float('nan'), float('nan'), float('nan')
    dbar = d.mean()
    # Newey-West variance estimator with lag h-1
    g0 = np.var(d, ddof=0)
    var_d = g0
    for k in range(1, h):
        gk = np.cov(d[:-k], d[k:], ddof=0)[0, 1] if n > k else 0.0
        var_d += 2 * (1 - k / h) * gk
    var_d = max(var_d, 1e-12)
    dm = dbar / np.sqrt(var_d / n)
    # HLN correction for small samples
    correction = np.sqrt((n + 1 - 2*h + h*(h-1)/n) / n)
    dm_hln = dm * correction
    p = 2 * (1 - stats.t.cdf(abs(dm_hln), df=n-1))
    return float(dm_hln), float(p), float(dbar)


def pesaran_timmermann(y_true, y_pred):
    '''Pesaran-Timmermann (1992) test of directional predictive accuracy.
    Two-sided p-value. Null: y_pred and y_true are independent in direction.

    The test statistic compares the realised hit rate to the hit rate
    expected under independence, scaled by an asymptotic standard error.
    We guard the variance to handle degenerate cases (unanimous predictions
    or labels) where the closed-form variance becomes 0 or negative due
    to finite-sample effects; in those degenerate cases we return p=NaN.
    '''
    y_true = np.asarray(y_true).astype(float)
    y_pred = np.asarray(y_pred).astype(float)
    n = len(y_true)
    if n < 10:
        return float('nan'), float('nan')
    P = float((y_true == y_pred).mean())
    py = float(y_true.mean()); pp = float(y_pred.mean())
    # If either side is degenerate (all 0 or all 1), test is undefined
    if py in (0.0, 1.0) or pp in (0.0, 1.0):
        return float('nan'), float('nan')
    P_star = py * pp + (1 - py) * (1 - pp)
    var_P      = (P_star * (1 - P_star)) / n
    var_P_star = ((2*py - 1)**2 * pp * (1 - pp) +
                  (2*pp - 1)**2 * py * (1 - py) +
                  4 * py * pp * (1 - py) * (1 - pp)) / n
    var = var_P - var_P_star
    if var <= 1e-10:
        # Variance vanishes: typically perfect or near-perfect predictions.
        # Fall back to a conservative two-sample proportion-test approximation.
        if abs(P - P_star) < 1e-6:
            return 0.0, 1.0
        return float('inf') if P > P_star else float('-inf'), 0.0
    PT = (P - P_star) / np.sqrt(var)
    # Cap |PT| at 50 to avoid numerical overflow in cdf for very strong signals
    PT_clipped = float(np.clip(PT, -50.0, 50.0))
    p = 2 * (1 - stats.norm.cdf(abs(PT_clipped)))
    return float(PT), float(p)


### 8.3 Apply to all model pairs against the strongest baseline

We compare each model against the best non-CD-GSHA classical/deep baseline (chosen by validation F1) — this is the contrast the paper actually argues about.


In [ ]:
# Build per-example forecast loss series for each model on the test set
y_te = y_all[te_idx]

def per_example_loss(y_true, y_prob, threshold):
    '''Squared-loss per example; standard for DM-style tests on probability
    forecasts. We use Brier loss (y - p)^2 which is a proper scoring rule.'''
    return (y_true.astype(float) - y_prob)**2

def per_example_pred(y_prob, threshold):
    return (y_prob > threshold).astype(int)

# Pick the strongest non-CD-GSHA baseline by validation F1
non_cdgsha = {n: r for n, r in results.items() if n != 'CD-GSHA'}
ref_name   = max(non_cdgsha, key=lambda n: non_cdgsha[n]['val_f1'])
print(f'Reference baseline (by val F1): {ref_name}')

ref_loss = per_example_loss(y_te, results[ref_name]['test_prob'],
                            results[ref_name]['threshold'])
ref_pred = per_example_pred(results[ref_name]['test_prob'],
                            results[ref_name]['threshold'])

sig_rows = []
for name, r in results.items():
    loss  = per_example_loss(y_te, r['test_prob'], r['threshold'])
    pred  = per_example_pred(r['test_prob'], r['threshold'])
    dm_stat, dm_p, dm_diff = diebold_mariano(loss, ref_loss, h=1)
    pt_stat, pt_p          = pesaran_timmermann(y_te, pred)
    sig_rows.append({
        'Model':       name,
        'DM stat':     dm_stat,
        'DM p-value':  dm_p,
        'Mean Δ loss': dm_diff,
        'PT stat':     pt_stat,
        'PT p-value':  pt_p,
        'F1':          r['f1'],
    })

sig_df = pd.DataFrame(sig_rows).set_index('Model')
print()
print(f'Diebold-Mariano vs {ref_name} (negative DM = lower loss; PT p < 0.05 = directional skill)')
print('=' * 92)
print(sig_df.to_string(float_format=lambda v: f'{v:+.4f}' if abs(v) < 100 else f'{v:.2e}'))


---

## 9. Full Backtesting

ICAIF reviewers expect more than F1/accuracy. We run a long-short backtest with confidence-weighted position sizing on the held-out test set, and report Sharpe ratio, Sortino ratio, max drawdown, turnover, hit rate, and a transaction-cost sensitivity sweep.

### 9.1 Strategy specification

- **Universe:** Single commodity (wheat for this notebook; corn/oil run separately).
- **Signal:** Predicted probability $\hat{p}_t = \Pr[y_{t+1} = 1]$ from each model.
- **Sizing:** Confidence-weighted. Position $w_t = 2(\hat{p}_t - 0.5)$ clipped to $[-1, 1]$. Long if $\hat{p}_t > 0.5$, short if $\hat{p}_t < 0.5$, magnitude proportional to conviction.
- **Rebalance:** Daily, at the close.
- **Returns:** $r^{\text{strat}}_{t+1} = w_t \cdot r_{t+1} - c \cdot |w_t - w_{t-1}|$ where $r_{t+1}$ is the next-day log return and $c$ is the round-trip transaction cost in basis points.
- **Cost sensitivity:** Sweep $c \in \{0, 1, 3, 5\}$ bps.

### 9.2 No-leakage protocol

- Strategy parameters (sizing function, cost level, sweep grid) are fixed *before* any test predictions are generated.
- Only ensembled test probabilities are used; per-seed predictions never enter the backtest.
- Test set probabilities go in; trading P&L comes out. No tuning loop on test data.


In [ ]:
def confidence_position(prob):
    '''Confidence-weighted long-short position in [-1, 1].'''
    return np.clip(2 * (prob - 0.5), -1.0, 1.0)


def backtest_strategy(test_prob, test_dates, cost_bps=1.0):
    '''Run long-short backtest. test_prob aligned to test_dates.

    Returns dict of per-day strategy returns + summary stats.
    '''
    # Get next-day log returns aligned to test_dates
    test_dates_pd = pd.to_datetime(test_dates)
    # Index price_df by date string
    price_lookup = dict(zip(
        price_df.index.strftime('%Y-%m-%d').tolist(),
        price_df['Return'].values
    ))
    # Next-day return for each test date d is the return on day d+1
    # We have the label y as direction(t+1), and we're trading at close of t
    # So strategy_return_{t+1} = w_t * (next-day return after t)
    next_day_rets = []
    for d in test_dates:
        # The return associated with the label is the day t+1 return.
        # In our setup, dates_all[i] is the LAST day of window;
        # the label y_all[i] = (P_{t+1} > P_t), so the relevant return
        # for trading from close-of-t to close-of-(t+1) is Return on day t+1.
        d_pd = pd.to_datetime(d)
        # Find next trading day after d
        future_dates = price_df.index[price_df.index > d_pd]
        if len(future_dates) == 0:
            next_day_rets.append(0.0)
        else:
            next_d = future_dates[0]
            next_day_rets.append(float(price_df.loc[next_d, 'Return']))
    next_day_rets = np.array(next_day_rets)

    # Positions
    w = confidence_position(test_prob)
    # Strategy gross return
    gross = w * next_day_rets
    # Transaction cost: c * |Δw|
    w_prev = np.concatenate([[0.0], w[:-1]])
    turnover_per_day = np.abs(w - w_prev)
    cost = (cost_bps / 1e4) * turnover_per_day
    net = gross - cost

    # Cumulative returns
    cum_gross = np.cumprod(1 + gross)
    cum_net   = np.cumprod(1 + net)

    # Sharpe / Sortino (annualised, 252 trading days)
    def _sharpe(r):
        if r.std() < 1e-12: return 0.0
        return (r.mean() / r.std()) * np.sqrt(252)
    def _sortino(r):
        downside = r[r < 0]
        if len(downside) == 0 or downside.std() < 1e-12: return 0.0
        return (r.mean() / downside.std()) * np.sqrt(252)
    def _max_dd(cum):
        peak = np.maximum.accumulate(cum)
        return float((cum - peak).min() / peak.max() if peak.max() > 0 else 0.0)

    return dict(
        cost_bps     = cost_bps,
        ret_gross    = gross,
        ret_net      = net,
        cum_gross    = cum_gross,
        cum_net      = cum_net,
        sharpe_gross = _sharpe(gross),
        sharpe_net   = _sharpe(net),
        sortino_net  = _sortino(net),
        max_dd_net   = _max_dd(cum_net),
        ann_ret_net  = float((cum_net[-1] ** (252 / max(len(net), 1)) - 1)),
        ann_vol_net  = float(net.std() * np.sqrt(252)),
        turnover_avg = float(turnover_per_day.mean()),
        hit_rate     = float((np.sign(gross) > 0).mean()),
        n_days       = len(net),
    )


### 9.3 Run backtests for all models at multiple cost levels


In [ ]:
COST_GRID = [0.0, 1.0, 3.0, 5.0]
backtest_results = {}

for name, r in results.items():
    backtest_results[name] = {}
    for c in COST_GRID:
        bt = backtest_strategy(r['test_prob'], test_dates, cost_bps=c)
        backtest_results[name][c] = bt

# Summary table at 1bp (the headline cost level)
HEADLINE_COST = 1.0
bt_rows = []
for name, by_cost in backtest_results.items():
    bt = by_cost[HEADLINE_COST]
    bt_rows.append({
        'Model':       name,
        'Sharpe (net)':    bt['sharpe_net'],
        'Sortino (net)':   bt['sortino_net'],
        'Ann. Ret':        bt['ann_ret_net'],
        'Ann. Vol':        bt['ann_vol_net'],
        'Max DD':          bt['max_dd_net'],
        'Turnover':        bt['turnover_avg'],
        'Hit Rate':        bt['hit_rate'],
    })
bt_df = pd.DataFrame(bt_rows).set_index('Model')

print('=' * 92)
print(f'  BACKTEST — {COMMODITY.upper()} — cost = {HEADLINE_COST:.0f} bp round-trip')
print('=' * 92)
print(bt_df.to_string(float_format=lambda v: f'{v:+.4f}'))
print()
print(f'Best Sharpe:   {bt_df["Sharpe (net)"].idxmax()} ({bt_df["Sharpe (net)"].max():.4f})')
print(f'Best Ann.Ret:  {bt_df["Ann. Ret"].idxmax()} ({bt_df["Ann. Ret"].max():.4f})')
print(f'Smallest DD:   {bt_df["Max DD"].idxmax()} ({bt_df["Max DD"].max():.4f})')


### 9.4 Transaction-cost sensitivity


In [ ]:
sens_rows = []
for name, by_cost in backtest_results.items():
    row = {'Model': name}
    for c in COST_GRID:
        row[f'Sharpe@{int(c)}bp'] = by_cost[c]['sharpe_net']
    sens_rows.append(row)
sens_df = pd.DataFrame(sens_rows).set_index('Model')
print('Sharpe ratio sensitivity to transaction cost:')
print('=' * 80)
print(sens_df.to_string(float_format=lambda v: f'{v:+.4f}'))


### 9.5 Equity curves


In [ ]:
sns.set_style('whitegrid')
fig, ax = plt.subplots(figsize=(13, 5.5))
test_dates_pd = pd.to_datetime(test_dates)
for name, by_cost in backtest_results.items():
    bt = by_cost[HEADLINE_COST]
    is_cdgsha = (name == 'CD-GSHA')
    is_classical = name in ('Logistic', 'XGBoost', 'ARIMA-X')
    color = '#d62728' if is_cdgsha else ('#999999' if is_classical else '#4c78a8')
    lw = 2.5 if is_cdgsha else 1.2
    alpha = 1.0 if is_cdgsha else 0.7
    ax.plot(test_dates_pd, bt['cum_net'], label=name, color=color,
            linewidth=lw, alpha=alpha)
ax.axhline(1.0, color='black', linestyle=':', linewidth=0.8)
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative return (net of 1bp cost)')
ax.set_title(f'{COMMODITY.title()} — long-short backtest equity curves')
ax.legend(loc='upper left', fontsize=9, ncol=2)
plt.tight_layout(); plt.savefig(f'{COMMODITY}_equity_curves.png', dpi=150, bbox_inches='tight')
plt.show()


---

## 10. Ablation Study

We isolate the contribution of each CD-GSHA component. Identical training protocol; the architecture is the only thing that varies.


In [ ]:
ABLATION_CLASSES = {
    'CD-GSHA':                    CDGSHA,
    'CD-GSHA - temporal priors':  CDGSHA_NoTemporal,
    'CD-GSHA - graph':            CDGSHA_NoGraph,
    'CD-GSHA - hyperbolic part':  CDGSHA_NoHyperbolic,
    'CD-GSHA - euclidean part':   CDGSHA_NoEuclidean,
    'CD-GSHA - price causal':     CDGSHA_NoPriceCausal,
    'CD-GSHA + multi-head':       CDGSHA_MultiHead,
    'CD-GSHA + sym graph':        GSHA_Symmetric,
}

print('=' * 80)
print(f' ABLATION STUDY — {COMMODITY.upper()} — {len(HPARAMS["seeds_ablation"])} seeds')
print('=' * 80)
print()

ablation_results = {}
for name, cls in ABLATION_CLASSES.items():
    if name == 'CD-GSHA' and 'CD-GSHA' in results:
        # Reuse from headline if it was run with the same seed pool
        # But headline used 20 seeds; ablations use 10. Re-train with 10 for fair comparison.
        pass
    ablation_results[name] = run_deep_experiment(cls, name, HPARAMS['seeds_ablation'], verbose=True)

# Build delta table
ref = ablation_results['CD-GSHA']
abl_rows = []
for name, r in ablation_results.items():
    abl_rows.append({
        'Variant':        name,
        'F1':             r['f1'],
        'MCC':            r['mcc'],
        'Bal Acc':        r['balanced_acc'],
        'AUC':            r['auc'],
        'd F1':           r['f1']            - ref['f1'],
        'd MCC':          r['mcc']           - ref['mcc'],
        'd BalAcc':       r['balanced_acc']  - ref['balanced_acc'],
    })
abl_df = pd.DataFrame(abl_rows).set_index('Variant')

print()
print('=' * 92)
print('  ABLATION RESULTS')
print('=' * 92)
print(abl_df.to_string(float_format=lambda v: f'{v:+.4f}' if abs(v) < 0.1 else f'{v:.4f}'))


### 10.1 Ablation visualisation


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
abl_plot = abl_df.drop('CD-GSHA', errors='ignore')
x = np.arange(len(abl_plot)); width = 0.27
ax.bar(x - width, abl_plot['d F1'],     width, color='#d62728', label='Δ F1')
ax.bar(x,         abl_plot['d MCC'],    width, color='#4c78a8', label='Δ MCC')
ax.bar(x + width, abl_plot['d BalAcc'], width, color='#2ca02c', label='Δ Bal-Acc')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(abl_plot.index, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Δ vs full CD-GSHA')
ax.set_title(f'{COMMODITY.title()} — ablation: contribution of each component')
ax.legend(loc='best', fontsize=9)
plt.tight_layout(); plt.savefig(f'{COMMODITY}_ablation.png', dpi=150, bbox_inches='tight')
plt.show()


### 10.2 Backtest of the symmetric-graph ablation

The most important comparison: full CD-GSHA (directional) vs. CD-GSHA with the symmetric graph swapped in. If directionality contributes, the directional version should backtest better. If not, the simpler symmetric version is preferable.


In [ ]:
key_pairs = [
    ('CD-GSHA',              ABLATION_CLASSES['CD-GSHA']),
    ('CD-GSHA + sym graph',  ABLATION_CLASSES['CD-GSHA + sym graph']),
]

print('Direct comparison: directional vs symmetric graph (backtest at 1bp)')
print('=' * 80)
for name, _cls in key_pairs:
    bt = backtest_strategy(ablation_results[name]['test_prob'], test_dates, cost_bps=1.0)
    print(f'  {name:24s}  Sharpe={bt["sharpe_net"]:+.4f}  Ann.Ret={bt["ann_ret_net"]:+.4f}  '
          f'MaxDD={bt["max_dd_net"]:+.4f}')

# DM test on directional vs symmetric
loss_dir = (y_te - ablation_results['CD-GSHA']['test_prob'])**2
loss_sym = (y_te - ablation_results['CD-GSHA + sym graph']['test_prob'])**2
dm_stat, dm_p, dm_diff = diebold_mariano(loss_dir, loss_sym, h=1)
print(f'\nDiebold-Mariano (directional vs symmetric): DM = {dm_stat:+.3f}, p = {dm_p:.4f}')
print('  (negative DM = directional has lower loss)')


---

## 11. Summary Visualisations and Paper-Ready Tables

### 11.1 Combined performance heatmap


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.4))
heat = results_df[['Accuracy', 'Balanced Acc', 'F1', 'MCC', 'AUC',
                   'Prec (Up)', 'Prec (Down)']].astype(float)
sns.heatmap(heat, annot=True, fmt='.3f', cmap='RdYlGn',
            cbar_kws={'shrink': 0.6}, ax=ax, linewidths=0.5, linecolor='white',
            vmin=heat.min().min() - 0.02, vmax=heat.max().max() + 0.02)
ax.set_title(f'{COMMODITY.title()} — per-model classification metrics')
plt.tight_layout(); plt.savefig(f'{COMMODITY}_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


### 11.2 Bar chart — primary metrics


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
palette = {n: ('#d62728' if n == 'CD-GSHA' else
              ('#999999' if n in ('Logistic', 'XGBoost', 'ARIMA-X') else '#4c78a8'))
           for n in results_df.index}
for ax, metric, baseline in zip(axes, ['F1', 'MCC', 'Balanced Acc'], [0.5, 0.0, 0.5]):
    vals = results_df[metric].values
    names = results_df.index.tolist()
    colors = [palette[n] for n in names]
    bars = ax.bar(range(len(vals)), vals, color=colors, edgecolor='white')
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(names, rotation=25, ha='right', fontsize=9)
    ax.axhline(baseline, ls=':', color='gray', lw=0.8, label=f'random = {baseline}')
    ax.set_ylabel(metric)
    lo = min(min(vals) - 0.05, baseline - 0.05); hi = max(max(vals) + 0.05, baseline + 0.05)
    ax.set_ylim(lo, hi)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.005, f'{v:.3f}',
                ha='center', fontsize=8)
    ax.legend(loc='lower right', fontsize=8); ax.set_title(metric)
plt.tight_layout(); plt.savefig(f'{COMMODITY}_primary.png', dpi=150, bbox_inches='tight')
plt.show()


### 11.3 Paper-ready table

The combined classification + significance + backtesting table that goes into the paper.


In [ ]:
# Build paper table
paper_rows = []
for name, r in results.items():
    bt = backtest_results[name][HEADLINE_COST]
    sig = sig_df.loc[name]
    paper_rows.append({
        'Model':           name,
        'F1':              r['f1'],
        'MCC':             r['mcc'],
        'BalAcc':          r['balanced_acc'],
        'AUC':             r['auc'],
        'Sharpe':          bt['sharpe_net'],
        'AnnRet':          bt['ann_ret_net'],
        'MaxDD':           bt['max_dd_net'],
        'PT p-val':        sig['PT p-value'],
        'DM p-val (vs ref)':sig['DM p-value'],
    })
paper_df = pd.DataFrame(paper_rows).set_index('Model')

print('=' * 110)
print(f'  PAPER TABLE — {COMMODITY.upper()}')
print('=' * 110)
print(paper_df.to_string(float_format=lambda v: f'{v:+.4f}' if isinstance(v, float) else str(v)))
print()
print(f'(DM p-values are vs reference baseline = {ref_name}; PT tests directional skill)')

# Save as CSV for the paper
paper_df.to_csv(f'{COMMODITY}_paper_table.csv')
print(f'\nSaved: {COMMODITY}_paper_table.csv')


---

## 12. Discussion

This section is a placeholder structure — fill in with your actual findings after running on all three commodities.

### What the empirical evidence supports

The honest answer depends on the numbers you observe:

1. **If CD-GSHA wins on F1 *and* Sharpe consistently across wheat/corn/oil:** the architecture is contributing real signal, and the paper makes a clean contribution claim.
2. **If CD-GSHA wins on F1 but loses on Sharpe (or vice versa):** the paper becomes a study of how statistical accuracy and economic value can diverge — itself an interesting and publishable finding for ICAIF.
3. **If CD-GSHA does not win:** the paper reports honestly that on weak-signal commodity prediction, simpler architectures (cross-attention, XGBoost) are competitive with more complex ones. This is also publishable — negative results are valuable, and the rigorous evaluation protocol is itself a contribution.

### Expected interpretation patterns

- **Temporal priors ablation matters more than the graph itself:** suggests it's the temporal structure, not the news graph specifically, that is doing the work.
- **Hyperbolic ablation is small or even slightly improves:** suggests the data may not have enough hierarchical structure for hyperbolic geometry to help; the residual contribution is from the shape regularisation effect.
- **Symmetric-graph ablation matches directional:** would imply the directionality is not a meaningful contribution on this data — the simpler architecture wins.
- **Multi-head is worse than single-head:** confirms our design choice on weak-signal tasks.

### Limitations

- Single-commodity per notebook (multi-commodity comparison happens by running on all three and aggregating).
- 16-D FinBERT embeddings are low-capacity; a larger encoder may change conclusions.
- News density varies dramatically across commodities (~50% for wheat, ~95% for oil). Conclusions about graph behaviour with sparse news may not transfer.
- Daily frequency only. Intraday or weekly horizons may favour different architectures.
- Trading evaluation uses a simple confidence-weighted strategy with no risk management. Real implementation would add stops, position limits, and volatility targeting.

---

## References (cited inline above)

- Defferrard, Bresson, Vandergheynst (2016). Convolutional Neural Networks on Graphs with Fast Localized Spectral Filtering. NeurIPS.
- Tong et al. (2020). Digraph Inception Convolutional Networks. NeurIPS.
- Zhang et al. (2021). MagNet: A Neural Network for Directed Graphs. NeurIPS.
- Rey, Ajorlou, Mateos (2025). Directed Acyclic Graph Convolutional Networks. arXiv:2506.12218.
- Veličković et al. (2018). Graph Attention Networks. ICLR.
- Nickel, Kiela (2017). Poincaré Embeddings for Learning Hierarchical Representations. NeurIPS.
- Sawhney et al. (2021). HyperStockGAT: Hyperbolic Graph Attention for Stock Selection.
- Lin et al. (2017). Focal Loss for Dense Object Detection. ICCV.
- Xiong et al. (2020). On Layer Normalization in the Transformer Architecture. ICML.
- Loshchilov, Hutter (2019). Decoupled Weight Decay Regularization. ICLR.
- Izmailov et al. (2018). Averaging Weights Leads to Wider Optima and Better Generalization. UAI.
- Diebold, Mariano (1995). Comparing Predictive Accuracy. JBES.
- Harvey, Leybourne, Newbold (1997). Testing the Equality of Prediction Mean Squared Errors. International Journal of Forecasting.
- Pesaran, Timmermann (1992). A Simple Nonparametric Test of Predictive Performance. JBES.
